# Getting Started with Polars

In [ ]:
from typing import TypeVar

import polars as pl
import polars.selectors as cs
from polars.testing import assert_frame_equal

In [ ]:
_ = pl.Config.set_tbl_width_chars(73)
_ = pl.Config.set_float_precision(2)
_ = pl.Config.set_tbl_cols(6)
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_formatting("UTF8_BORDERS_ONLY")
_ = pl.Config.set_tbl_hide_dtype_separator(True)

In [ ]:
sample = pl.DataFrame({'city_mpg': [20, 30]})
print(sample.select(city_mpg_plus_one=pl.col('city_mpg') + 1))

In [ ]:
df_ops = set(x for x in dir(pl.DataFrame()) if not x.startswith('_'))
lazy_ops = set(x for x in dir(pl.LazyFrame()) if not x.startswith('_'))
print(sorted(df_ops - lazy_ops))

In [ ]:
print(sorted(lazy_ops & df_ops))

In [ ]:
col = pl.col('sample')
col_ops = set(x for x in dir(col) if not x.startswith('_'))
print(sorted(col_ops))

In [ ]:
print(col
  .cast(pl.Int32)
  .clip(upper_bound=100)
  .mean()
)

In [ ]:
print(pl.col.a_column)

In [ ]:
path = "data/vehicles.csv"
pl.read_csv(path)

In [ ]:
pl.read_csv(path, null_values='NA')

In [ ]:
df = pl.read_csv(path, use_pyarrow=True)

In [ ]:
print(df)

In [ ]:
print(
    df.select('make', 'model', 'year')
    .glimpse(max_items_per_column=2, return_type='frame')
)

In [ ]:
path_clean = "data/clean.csv"
df.write_csv(path_clean)

In [ ]:
lazy = pl.scan_csv("data/clean.csv")
print(lazy)

In [ ]:
print(lazy
  .filter((pl.col('year') >= 1990) & (pl.col('year') < 2000))
  .select(['year', 'make', 'model'])
)

In [ ]:
print(lazy
  .filter((pl.col('year') >= 1990) & (pl.col('year') < 2000))
  .select(['year', 'make', 'model'])
  .explain(optimized=True)
)

In [ ]:
print(df.dtypes)

In [ ]:
print(df.schema)

In [ ]:
cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany', 'drive',
        'VClass', 'fuelType', 'barrels08', 'city08', 'highway08',
        'createdOn']

In [ ]:
print(df.select(cols))

In [ ]:
print(df
  .select(pl.col(cols))
  .select(pl.col(pl.Int64))
  .describe()
)

In [ ]:
print(df
  .select(pl.col(cols))
  .with_columns(pl.col('year').cast(pl.Int16),
                pl.col('cylinders').cast(pl.UInt8),
                pl.col('highway08').cast(pl.UInt8),
                pl.col('city08').cast(pl.UInt8))
)

In [ ]:
print(df.head(3)
  .select(pl.col(cols))
  .with_columns(pl.col('year').cast(pl.Int8))
)

In [ ]:
print(df
  .select(pl.col(cols))
  .with_columns(pl.col('year').cast(pl.Int16),
                pl.col('cylinders').cast(pl.UInt8),
                pl.col('highway08').cast(pl.UInt8),
                pl.col('city08').cast(pl.UInt8))
  .select(pl.col(pl.Float64))
  .sample(n=10, seed=42)
)

In [ ]:
print(df
  .select(pl.col(cols))
  .with_columns(pl.col('year').cast(pl.Int16),
                pl.col('cylinders').cast(pl.UInt8),
                pl.col('highway08').cast(pl.UInt8),
                pl.col('city08').cast(pl.UInt8),
                pl.col('displ').cast(pl.Float32),
                pl.col('barrels08').cast(pl.Float32),
                )
)

In [ ]:
print(df
 .select('trany')
)

In [ ]:
print(df
 .select('trany',
         pl.col('trany')
              .str.to_lowercase()
              .str.contains('automatic') 
              .alias('automatic'))         
)

In [ ]:
print(df
 .with_columns(pl.col('trany')
              .str.to_lowercase()
              .str.contains('automatic') 
              .alias('is_automatic'))         
)

In [ ]:
print(df
 .with_columns('trany',
     is_automatic=pl.col('trany')
         .str.to_lowercase()
         .str.contains('automatic')) 
)

In [ ]:
print(df
 .group_by('trany')
 .len()
 .sort(['len', 'trany'], descending=[True, False])          
)

In [ ]:
print(df
 .select(num_gears=pl.col('trany')
            .str.extract(r'(\d+)')
            .cast(pl.UInt8))
)

In [ ]:
print(df
 .select(num_gears=pl.col('trany')
            .str.extract(r'(\d+)')
            .cast(pl.UInt8)
            .unique())
)

In [ ]:
print(df
 .select('trany', num_gears=pl.col('trany')
            .str.extract(r'(\d+)')
            .cast(pl.UInt8))
 .unique(subset=['trany', 'num_gears'])
 .filter(pl.col('num_gears').is_null())          
)

In [ ]:
print(df
 .with_columns(
    is_automatic=pl.col('trany')
              .str.contains('Automatic'),             
    num_gears=pl.col('trany')
            .str.extract(r'(\d+)')
            .cast(pl.UInt8)
              ) 
)

In [ ]:
print(df
  .select(pl.col(cols))
  .with_columns(pl.col('year').cast(pl.Int16),
                pl.col('cylinders').cast(pl.UInt8),
                pl.col('highway08').cast(pl.UInt8),
                pl.col('city08').cast(pl.UInt8),
                pl.col('displ').cast(pl.Float32),
                pl.col('barrels08').cast(pl.Float32),
                )
  .select(pl.col(pl.String))
)

In [ ]:
print(df  
  .select(pl.col(cols))
  .with_columns(pl.col('year').cast(pl.Int16),
       pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
       pl.col(['displ', 'barrels08']).cast(pl.Float32),
       pl.col(['make', 'model', 'VClass', 'drive', 
               'fuelType']).cast(pl.Categorical),
       )
)

In [ ]:
size_enum = pl.Enum(['xsmall', 'small', 'medium', 'large', 'xlarge'])
values = ['small', 'xlarge', 'small', 'xsmall']

print(pl.DataFrame({'size1_str': values})
 .with_columns(size1_cat=pl.col('size1_str').cast(pl.Categorical),
               size1_enum=pl.col('size1_str').cast(size_enum))
 .with_columns(size1_cat_phys=pl.col('size1_cat').to_physical(),
               size1_enum_phys=pl.col('size1_enum').to_physical())
)

In [ ]:
car_classes = ['Minicompact Cars', 'Subcompact Cars', 'Compact Cars',
               'Midsize Cars', 'Large Cars']
print(df
  .filter(pl.col('VClass').is_in(car_classes))
  .select('VClass')
  .unique()
  .sort('VClass')
)

In [ ]:
vclass_type = pl.Enum(car_classes)
print(df
  .filter(pl.col('VClass').is_in(car_classes))
  .select(pl.col('VClass').cast(vclass_type))
  .select(pl.col('VClass'), vtype=pl.col('VClass').to_physical())
  .unique()
  .sort('VClass')
)

In [ ]:
print(df
  .select(pl.col(cols))
  .with_columns(pl.col('year').cast(pl.Int16),
       pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
       pl.col(['displ', 'barrels08']).cast(pl.Float32),
       pl.col(['make', 'model', 'VClass', 'drive', 
               'fuelType']).cast(pl.Categorical),
       is_automatic=pl.col('trany')
              .str.contains('Automatic'),
       num_gears=pl.col('trany')
             .str.extract(r'(\d+)')
             .cast(pl.UInt8))                      
)

In [ ]:
created_date = (pl.col('createdOn')
    .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
    .str.to_date('%b %d %Y'))
print(df.select('createdOn', created_date.alias('created_date')))

In [ ]:
Frame = TypeVar("Frame", pl.DataFrame, pl.LazyFrame)

def tweak_auto(df: Frame) -> Frame:
   cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany', 
          'drive', 'VClass', 'fuelType', 'barrels08', 'city08', 
          'highway08', 'createdOn']
   return (df
    .select(pl.col(cols))
    .with_columns(pl.col('year').cast(pl.Int16),
       pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
       pl.col(['displ', 'barrels08']).cast(pl.Float32),
       pl.col(['make', 'model', 'VClass', 'drive', 
               'fuelType']).cast(pl.Categorical),
       pl.col('createdOn')
           .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
           .str.to_date('%b %d %Y'),
       is_automatic=pl.col('trany')                    
              .str.contains('Automatic'),
       num_gears=pl.col('trany')
             .str.extract(r'(\d+)')
             .cast(pl.UInt8))    
    )

In [ ]:
print(tweak_auto(df))

In [ ]:
tweak_auto(df).estimated_size(unit='mb')

In [ ]:
df.select(cols).estimated_size(unit='mb')

In [ ]:
print(tweak_auto(df).describe())

In [ ]:
print(tweak_auto(df)
 .select(cs.numeric())
 .corr()
 .pipe(lambda df_: df_.insert_column(0, pl.Series('names', df_.columns)))
)

In [ ]:
print(tweak_auto(df)
 .select(pl.corr('year', 'displ'))
)

In [ ]:
print(tweak_auto(df)
 .to_pandas()
 .corr(method='spearman', numeric_only=True)
 # convert back to polars
 .pipe(pl.from_pandas, include_index=True)
)

In [ ]:
lazy = pl.scan_csv('data/clean.csv')
cleaned_lazy = tweak_auto(lazy).collect()
print(cleaned_lazy)

# The CLEANER Framework

In [ ]:
from typing import TypeVar

import polars as pl
from polars.testing import assert_frame_equal

In [ ]:
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_cols(8)
_ = pl.Config.set_tbl_width_chars(80)
_ = pl.Config.set_float_precision(2)

In [ ]:
path = 'data/vehicles.csv'
# Capture: load the original data
raw = pl.read_csv(path, use_pyarrow=True)
raw.shape

In [ ]:
Frame = TypeVar("Frame", pl.DataFrame, pl.LazyFrame)

def tweak_auto(df: Frame) -> Frame:
   cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany', 
          'drive', 'VClass', 'fuelType', 'barrels08', 'city08', 
          'highway08', 'createdOn']
   return (df  
    .select(pl.col(cols))  
    .with_columns(pl.col('year').cast(pl.Int16),  
       pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
       pl.col(['displ', 'barrels08']).cast(pl.Float32),
       pl.col(['make', 'model', 'VClass', 'drive', 
               'fuelType']).cast(pl.Categorical),
       pl.col('createdOn')
           .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
           .str.to_date('%b %d %Y'),
       is_automatic=pl.col('trany')  
              .str.contains('Automatic'),
       num_gears=pl.col('trany')
             .str.extract(r'(\d+)')
             .cast(pl.UInt8))    
    )

In [ ]:
print(tweak_auto(raw))

In [ ]:
cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany', 'drive',
  'VClass', 'fuelType', 'barrels08', 'city08', 'highway08', 'createdOn']

print(raw  # Capture: start with the original data
    .select(pl.col(cols))  # Limit: select input columns
)

In [ ]:
print(raw
    .select('city08', 'highway08')
    .rename({'city08': 'city_mpg', 'highway08': 'highway_mpg'})
    .head()
)

In [ ]:
print(raw  # Capture: start with the original data
    .select(pl.col(cols))  # Limit: select input columns
    # Edit: keep the original column names
    .select(  # Audit: inspect numeric ranges
        pl.col('year', 'cylinders', 'city08', 'highway08')
            .min().name.suffix('_min'),
        pl.col('year', 'cylinders', 'city08', 'highway08')
            .max().name.suffix('_max'),
    )
)

In [ ]:
print(raw  # Capture: start with the original data
    .select(pl.col(cols))  # Limit: select input columns
    # Edit: keep the original column names
    .with_columns(  # Audit: apply the chosen types
        pl.col('year').cast(pl.Int16),
        pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
        pl.col(['displ', 'barrels08']).cast(pl.Float32),
        pl.col(['make', 'model', 'VClass', 'drive', 
            'fuelType']).cast(pl.Categorical),
        pl.col('createdOn')  # Audit: convert the text to a Date
          .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
          .str.to_date('%b %d %Y'),
    )
    .schema  # Audit: inspect the resulting types
)

In [ ]:
print(raw  # Capture: start with the original data
    .select(pl.col(cols))  # Limit: select input columns
    # Edit: keep the original column names
    .with_columns(  # Audit: apply the chosen types
        pl.col('year').cast(pl.Int16),
        pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
        pl.col(['displ', 'barrels08']).cast(pl.Float32),
        pl.col(['make', 'model', 'VClass', 'drive', 
            'fuelType']).cast(pl.Categorical),
        pl.col('createdOn')  # Audit: convert the text to a Date
          .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
          .str.to_date('%b %d %Y'),
    )
    .null_count()  # Null: count missing values
)

In [ ]:
print(raw  # Capture: start with the original data
    .select(pl.col(cols))  # Limit: select input columns
    # Edit: keep the original column names
    .with_columns(  # Audit: apply the chosen types
        pl.col('year').cast(pl.Int16),
        pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
        pl.col(['displ', 'barrels08']).cast(pl.Float32),
        pl.col(['make', 'model', 'VClass', 'drive', 
            'fuelType']).cast(pl.Categorical),
        pl.col('createdOn')  # Audit: convert the text to a Date
          .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
          .str.to_date('%b %d %Y'),
    )
    .select(  # Null: inspect missing and empty values
        missing_transmission=pl.col('trany').is_null().sum(),
        empty_transmission=(pl.col('trany') == '').sum(),
    )
)

In [ ]:
print(raw  # Capture: start with the original data
    .select(pl.col(cols))  # Limit: select input columns
    # Edit: keep the original column names
    .with_columns(  # Audit: apply the chosen types
        pl.col('year').cast(pl.Int16),
        pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
        pl.col(['displ', 'barrels08']).cast(pl.Float32),
        pl.col(['make', 'model', 'VClass', 'drive', 
            'fuelType']).cast(pl.Categorical),
        pl.col('createdOn')  # Audit: convert the text to a Date
          .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
          .str.to_date('%b %d %Y'),
    )  # Null: keep missing values and empty strings
    .with_columns(  # Engineer: extract transmission features
        is_automatic=pl.col('trany').str.contains('Automatic'),
        num_gears=pl.col('trany').str.extract(r'(\d+)').cast(pl.UInt8),
    )
    # Engineer: inspect the new columns
    .select('trany', 'is_automatic', 'num_gears')  
    .unique()
    .sort('trany')
)

In [ ]:
print(raw  # Capture: start with the original data
    .select(pl.col(cols))  # Limit: select input columns
    # Edit: keep the original column names
    .with_columns(  # Audit: apply the chosen types
        pl.col('year').cast(pl.Int16),
        pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
        pl.col(['displ', 'barrels08']).cast(pl.Float32),
        pl.col(['make', 'model', 'VClass', 'drive', 
            'fuelType']).cast(pl.Categorical),
        pl.col('createdOn')  # Audit: convert the text to a Date
          .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
          .str.to_date('%b %d %Y'),
    )  # Null: keep missing values and empty strings
    .with_columns(  # Engineer: extract transmission features
        is_automatic=pl.col('trany').str.contains('Automatic'),
        num_gears=pl.col('trany').str.extract(r'(\d+)').cast(pl.UInt8),
    )
    # Restrict: choose output columns
    .select(cols + ['is_automatic', 'num_gears'])  
)

In [ ]:
def tweak_auto_cleaner(df: Frame) -> Frame:
    cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany',
            'drive', 'VClass', 'fuelType', 'barrels08', 'city08',
            'highway08', 'createdOn']
    return (df  # Capture: start with the original data
        .select(pl.col(cols))  # Limit: select input columns
        # Edit: keep the original column names
        .with_columns(  # Audit: apply the chosen types
            pl.col('year').cast(pl.Int16),
            pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
            pl.col(['displ', 'barrels08']).cast(pl.Float32),
            pl.col(['make', 'model', 'VClass', 'drive', 
                'fuelType']).cast(pl.Categorical),
            pl.col('createdOn')  # Audit: convert the text to a Date
              .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
              .str.to_date('%b %d %Y'),
        )  # Null: keep missing values and empty strings
        .with_columns(  # Engineer: extract transmission features
            is_automatic=pl.col('trany').str.contains('Automatic'),
            num_gears=pl.col('trany').str.extract(r'(\d+)')
            .cast(pl.UInt8),
        )
        # Restrict: choose output columns
        .select(cols + ['is_automatic', 'num_gears'])  
    )

In [ ]:
assert_frame_equal(tweak_auto_cleaner(raw), tweak_auto(raw))

In [ ]:
assert isinstance(tweak_auto_cleaner(raw.lazy()), pl.LazyFrame)
assert_frame_equal(tweak_auto_cleaner(raw.lazy()).collect(),
    tweak_auto(raw))

In [ ]:
parquet_path = 'data/vehicles-cleaned.parquet'
tweak_auto_cleaner(raw).write_parquet(parquet_path)

In [ ]:
cleaned = pl.read_parquet(parquet_path)
assert_frame_equal(cleaned, tweak_auto(raw), check_dtypes=True,
    check_exact=True)

# Data Manipulation

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import xgboost as xgb

In [ ]:
_ = pl.Config.set_tbl_width_chars(80)
_ = pl.Config.set_float_precision(2)
_ = pl.Config.set_tbl_cols(6)
#pl.Config.set_tbl_formatting('ASCII_BORDERS_ONLY_CONDENSED')
_ = pl.Config.set_tbl_formatting('UTF8_BORDERS_ONLY')
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_hide_dtype_separator(True)

In [ ]:
parquet_path = 'data/vehicles-cleaned.parquet'
autos = pl.read_parquet(parquet_path)

In [ ]:
print(autos)

In [ ]:
# Reuse the 2019 records in the measurement examples below.
vehicles_2019 = autos.filter(pl.col('year') == 2019)

In [ ]:
print(autos
  .select(pl.col(['highway08', 'city08']),
          mpg_ratio=(pl.col('highway08') / pl.col('city08')))
          
 )

In [ ]:
print(autos
  .with_columns(mpg_ratio=pl.col('highway08') / pl.col('city08'))
 )

In [ ]:
print(autos
 .select('year', 'make', 'model', engine=pl.struct('displ', 'cylinders')))

In [ ]:
print(autos
 .select('year', 'make', 'model', engine=pl.struct('displ', 'cylinders'))
 .with_columns(pl.col('engine').struct.eval(
     pl.field('cylinders'),
     displacement_cc=(pl.field('displ') * 1_000).round(0)
     .cast(pl.Int32))))

In [ ]:
print(autos
 .select('year', 'make', 'model', engine=pl.struct('displ', 'cylinders'))
 .with_columns(pl.col('engine').struct.with_fields(
     six_or_more=pl.field('cylinders') >= 6)))

In [ ]:
print(autos
 .select('year', 'make', 'model', engine=pl.struct('displ', 'cylinders'))
 .with_columns(pl.col('engine').struct.drop(['displ']))
 .unnest('engine', separator='_'))

In [ ]:
print(autos
 .drop('createdOn')
 .columns
)

In [ ]:
final_cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany', 
  'drive', 'VClass', 'fuelType', 'barrels08', 'city08', 'highway08', 
  'is_automatic', 'num_gears']

print(autos
 .select(final_cols)
)

In [ ]:
print(autos
  .select(pl.all()
           .exclude(['createdOn', 'barrels08'])
           .name.suffix('_auto'))
)

In [ ]:
import polars.selectors as cs
print(autos
 .select(cs.integer() & cs.by_name('year', 'cylinders', 'city08',
     'highway08'))
)

In [ ]:
print(autos
 .select(cs.numeric() - cs.by_name('year'))
)

In [ ]:
no_dates_or_bools = (cs.all() - cs.temporal() - cs.boolean())
print(no_dates_or_bools)

In [ ]:
print(cs.expand_selector(autos, no_dates_or_bools))

In [ ]:
final_cols = ['year', 'make', 'model', 'city_mpg', 'highway_mpg']
print(autos
 .with_columns(pl.col('city08').alias('city_mpg'),
               highway_mpg=pl.col('highway08'))
 .select(final_cols)
 )

In [ ]:
final_cols = ['year', 'make', 'model', 'city_mpg', 'highway_mpg']
print(autos
 .with_columns(
         highway_mpg=pl.col('highway08'),
         pl.col('city08').alias('city_mpg'))
 .select(final_cols)
 )

In [ ]:
final_cols = ['year', 'make', 'model', 'city_mpg', 'highway_mpg']
print(autos
 .rename({'city08':'city_mpg',
          'highway08':'highway_mpg'})
 .select(final_cols)
 )

In [ ]:
print(autos
 .rename(lambda col: col.upper())
 )

In [ ]:
print(autos.select(pl.col('city08', 'highway08')
    .name.replace(pattern='08$', value='_mpg')))

In [ ]:
print(autos
 .select(pl.col('city08', 'highway08').mean().name.suffix('_mean'))
)

In [ ]:
print(autos
 .select(pl.col('city08', 'highway08').mean().name.suffix('_mean'))
 .select(cs.ends_with('_mean').name.strip_suffix('_mean'))
)

In [ ]:
print(autos
 .select(pl.col('city08', 'highway08').mean().name.prefix('summary_'))
)

In [ ]:
print(autos
 .with_row_index('source_position')
 .sort('city08', descending=True)
 .select('source_position', 'make', 'model', 'city08')
)

In [ ]:
print(autos
 .filter(pl.col('year') == 2019)
 .sort(['make', 'city08', 'model'], descending=[False, True, False])
 .with_columns(position=pl.row_index().over('make'))
 .filter(pl.col('position') < 3)
 .select('make', 'model', 'city08', 'position'))

In [ ]:
print(autos
 .filter((pl.col('year') == 2019) & pl.col('make').is_in(['Volvo', 
            'Nissan', 'Tesla']))
 .group_by('make')
 .agg(mean_displ=pl.col('displ').mean(), std_displ=pl.col('displ').std())
 .sort('make'))

In [ ]:
displ_z = ((pl.col('displ') - pl.col('displ').mean().over('make')) 
    / pl.col('displ').std().over('make'))
print(autos
 .filter((pl.col('year') == 2019) & pl.col('make').is_in(['Volvo',
            'Nissan', 'Tesla']))
 .with_columns(displ_z=displ_z)
 .select('year', 'make', 'model', 'displ', 'displ_z')
 )

In [ ]:
print(autos
 .filter((pl.col('year') == 2019) & pl.col('make').is_in(['Volvo', 
            'Nissan', 'Tesla']))
 .with_columns(displ_z=displ_z)
 .group_by('make')
 .agg(
     total=pl.col('displ_z').len(),
     missing=pl.col('displ_z').null_count(),
     nan=pl.col('displ_z').is_nan().sum(),
     finite=pl.col('displ_z').is_finite().sum()
 )
 .sort('make')
)

In [ ]:
print(autos
 .select('year', 'make', 'fuelType', 'displ',
         mean_displ=pl.col('displ').mean().over('year', 'make',
         'fuelType'))
 )

In [ ]:
print(autos
 .with_columns(mean_displ=pl.col('displ').mean().over('year', 
        'make', 'fuelType'))
 .filter(pl.col('displ').is_null())
 .select('year', 'make', 'model', 'fuelType', 'displ', 'mean_displ')
 .sort('fuelType', 'year', 'make')
)

In [ ]:
print(autos
 .with_columns(mean_displ=pl.col('displ').mean().over('year', 
        'make', 'fuelType'))
 .filter(pl.col('displ').is_null())
 .filter(pl.col('fuelType') != 'Electricity')
 .select('year', 'make', 'model', 'fuelType', 'displ', 'mean_displ')
 .sort('fuelType', 'year', 'make')
)

In [ ]:
print((autos
 .with_columns(mean_displ=pl.col('displ').mean().over('year', 'make',
     'fuelType'))
 .with_columns(estimated_displ=pl.when(
     pl.col('displ').is_null()
     & (pl.col('fuelType') != 'Electricity')
     & pl.col('mean_displ').is_not_null())
     .then(pl.col('mean_displ'))
     .otherwise(pl.col('displ'))))
 .filter(pl.col('displ').is_null())
 .select('year', 'make', 'model', 'fuelType', 'displ', 'estimated_displ')
 .sort('fuelType', 'year', 'make'))

In [ ]:
print(vehicles_2019
 .with_columns(
    lower_mpg=pl.col('city08').quantile(.05).over('VClass'),
    upper_mpg=pl.col('city08').quantile(.95).over('VClass'))
 .with_columns(
    clipped_mpg=pl.col('city08').clip('lower_mpg', 'upper_mpg')
 )
 .filter(pl.col('city08') != pl.col('clipped_mpg'))
 .select('make', 'model', 'city08', 'lower_mpg', 'upper_mpg',
     'clipped_mpg'))

In [ ]:
print(vehicles_2019
 .with_columns(
    lower_mpg=pl.col('city08').quantile(.05).over('VClass'),
    upper_mpg=pl.col('city08').quantile(.95).over('VClass'))
 .with_columns(clipped_mpg=pl.col('city08').clip('lower_mpg',
     'upper_mpg'))
 .group_by('make').agg(original_mean=pl.col('city08').mean(),
                      clipped_mean=pl.col('clipped_mpg').mean())
 .with_columns(change=pl.col('clipped_mean') - pl.col('original_mean'))
 .filter(pl.col('change') != 0)
 .sort(pl.col('change').abs(), descending=True))

In [ ]:
print((autos
 .select('fuelType')
 .to_dummies()

))

In [ ]:
X = autos.select('displ', 'cylinders')
print(X)

In [ ]:
y = autos.get_column('city08')
print(y)

In [ ]:
reg = xgb.XGBRegressor(enable_categorical=True)
reg.fit(X, y)

In [ ]:
pred = reg.predict(X)
print(pred)

In [ ]:
print(X
  .with_columns(autos['make'], autos['model'],
      predition=pred, actual=y)
)

In [ ]:
def fit_mpg_trend(columns: list[pl.Series]) -> np.ndarray:
    year = columns[0].to_numpy()
    mpg = columns[1].to_numpy()
    trend = np.polynomial.Polynomial.fit(year, mpg, deg=1)
    return trend(year)

In [ ]:
print(autos
 .filter(pl.col('make') == 'Ford')
 .group_by('year')
 .agg(median_city_mpg=pl.col('city08').median())
 .sort('year')
 .with_columns(
     fitted_city_mpg=pl.map_batches(
         ['year', 'median_city_mpg'],
         fit_mpg_trend,
         return_dtype=pl.Float64,
     )
 )
)

# Combining DataFrames

In [ ]:
from pathlib import Path
import polars as pl
import polars.selectors as cs
import numpy as np

In [ ]:
_ = pl.Config.set_tbl_width_chars(80)
_ = pl.Config.set_float_precision(2)
_ = pl.Config.set_tbl_cols(6)
#pl.Config.set_tbl_formatting('ASCII_BORDERS_ONLY_CONDENSED')
_ = pl.Config.set_tbl_formatting('UTF8_BORDERS_ONLY')
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_hide_dtype_separator(True)

In [ ]:
autos = pl.read_parquet('data/vehicles-cleaned.parquet')
print(autos)

In [ ]:
vehicles_2018 = autos.filter(pl.col('year') == 2018)
vehicles_2019 = autos.filter(pl.col('year') == 2019)
print(vehicles_2018)

In [ ]:
print(vehicles_2019)

In [ ]:
print(vehicles_2018
 .vstack(vehicles_2019)
 .select('year', 'make', 'model', 'city08')
 )

In [ ]:
snacks = pl.DataFrame(
    {'id': [1, 2, 3], 'snack': ['tacos', 'cake', 'chips']},
    schema_overrides={'id': pl.Int32},
)
hats = pl.DataFrame({'id': [2, 4], 'hat': ['crown', 'sombrero']})
print(snacks)
print(hats)

In [ ]:
later_snacks = pl.DataFrame({'id': [2, 4], 'snack': ['cake', 'popcorn']})
print(pl.concat([snacks, later_snacks], how='vertical_relaxed'))

In [ ]:
print(pl.concat([snacks, hats], how='diagonal_relaxed'))

In [ ]:
print(pl.concat([snacks, hats.rename({'id': 'hat_id'})],
                how='horizontal_extend'))

In [ ]:
print(vehicles_2018
 .sort('city08')
 .merge_sorted(vehicles_2019.sort('city08'), key='city08',
     maintain_order=True)
 .select('year', 'make', 'model', 'city08')
)

In [ ]:
print(pl.concat([vehicles_2018, vehicles_2019])
 .sort('cylinders', nulls_last=True)
 .select('year', 'make', 'model', 'cylinders')
)

In [ ]:
print(vehicles_2018
 .sort('make', 'model')
 .merge_sorted(vehicles_2019.sort('make', 'model'), 
        key=['make', 'model'], maintain_order=True)
 .select('year', 'make', 'model', 'city08')
)

In [ ]:
chunked = pl.concat([vehicles_2018, vehicles_2019], rechunk=False)
(chunked['cylinders'].n_chunks(), chunked['cylinders'].chunk_lengths())

In [ ]:
contiguous = chunked.rechunk()
(contiguous['cylinders'].n_chunks(),
    contiguous['cylinders'].chunk_lengths())

In [ ]:
models_2018 = (vehicles_2018
 .filter(pl.col('make') == 'Ford')
 .group_by('model')
 .agg(mpg_2018=pl.col('city08').median())
 .sort('model'))
print(models_2018)

In [ ]:
models_2019 = (vehicles_2019
 .filter(pl.col('make') == 'Ford')
 .group_by('model')
 .agg(mpg_2019=pl.col('city08').median())
 .sort('model'))
print(models_2019)

In [ ]:
print(models_2018.join(models_2019, how='left'))

In [ ]:
print(models_2018
 .join(models_2019, on='model', how='left')
 .sort('model'))

In [ ]:
print(models_2018
 .join(models_2019, on='model', how='right')
 .sort('model'))

In [ ]:
print(models_2018
 .join(models_2019, on='model', how='inner')
 .with_columns(mpg_change=pl.col('mpg_2019') - pl.col('mpg_2018'))
 .sort('model'))

In [ ]:
print(models_2018
 .join(models_2019, on='model', how='full', coalesce=True)
 .sort('model'))

In [ ]:
print(models_2018
 .join(models_2019, on='model', how='semi')
 .sort('model'))

In [ ]:
print(models_2018
 .join(models_2019, on='model', how='anti')
 .sort('model'))

In [ ]:
print(models_2019.join(models_2018, on='model', how='anti').sort('model'))

In [ ]:
print(models_2018.join(models_2019, on='model',
    validate='1:1').sort('model'))

In [ ]:
print(models_2018.join(
 vehicles_2019.filter(pl.col('make') == 'Ford'),
 on='model', validate='1:1'))

In [ ]:
print(models_2018
 .join(vehicles_2019.filter(pl.col('make') == 'Ford'),
       on='model', validate='1:m')
 .select('model', 'mpg_2018', 'city08')
 .sort('model', 'city08'))

In [ ]:
print(vehicles_2019
 .filter(pl.col('make') == 'Ford')
 .filter(pl.col('model').is_duplicated())
 .select('model', 'city08')
 .sort('model', 'city08'))

In [ ]:
print(vehicles_2019
 .filter(pl.struct('year', 'make', 'model').is_duplicated())
 .select('year', 'make', 'model', 'displ', 'trany', 'city08')
 .sort('make', 'model', 'displ', 'trany'))

In [ ]:
all_models_2018 = (vehicles_2018
 .group_by('make', 'model')
 .agg(mpg_2018=pl.col('city08').median()))
all_models_2019 = (vehicles_2019
 .group_by('make', 'model')
 .agg(mpg_2019=pl.col('city08').median()))
print(all_models_2018
 .join(all_models_2019, on=cs.by_name('make', 'model'), validate='1:1')
 .with_columns(mpg_change=pl.col('mpg_2019') - pl.col('mpg_2018'))
 .sort('mpg_change', 'make', 'model', descending=[True, False, False]))

In [ ]:
print(autos.select('year', 'make', 'model', 'city08',
 model_median=pl.col('city08').median().over(['year', 'make', 'model'])))

In [ ]:
cylinders_2018 = vehicles_2018.group_by('cylinders').len(
    name='count_2018')
cylinders_2019 = vehicles_2019.group_by('cylinders').len(
    name='count_2019')
print(cylinders_2018
 .join(cylinders_2019, on='cylinders', how='full',
       coalesce=True, nulls_equal=False)
 .sort('cylinders', nulls_last=True))

In [ ]:
print(cylinders_2018
 .join(cylinders_2019, on='cylinders', how='full',
       coalesce=True, nulls_equal=True)
 .sort('cylinders', nulls_last=True))

In [ ]:
ford_2018_2019 = ((pl.col('make') == 'Ford') & 
    pl.col('year').is_between(2018, 2019))
edge_focus_ranger = (pl.col('model').cast(pl.String).is_in(['Edge AWD', 
    'Focus FWD', 'Ranger 2WD']))
print(autos
 .filter(ford_2018_2019)
 .filter(edge_focus_ranger)
 .select('model')
 .unique()
 .join(autos
    .filter(pl.col('year').is_between(2018, 2019))
    .select('year')
    .unique(), how='cross')
 .sort('model', 'year'))

In [ ]:
print(autos
    .filter(ford_2018_2019)
    .group_by('model', 'year')
    .agg(city_mpg=pl.col('city08').median()))

In [ ]:
print(autos
 .filter(ford_2018_2019)
 .filter(edge_focus_ranger)
 .select('model').unique()
 .join(autos.filter(pl.col('year').is_between(2018, 2019))
       .select('year').unique(), how='cross')
 .join(autos.filter(ford_2018_2019)
       .group_by('model', 'year').agg(city_mpg=pl.col('city08').median()),
       on=['model', 'year'], how='left', validate='1:1')
 .sort('model', 'year'))

In [ ]:
trucks_ford_2018 = (autos
 .with_row_index('configuration')
 .filter((pl.col('make') == 'Ford') & (pl.col('year') == 2018)
         & (pl.col('model') == 'F150 Pickup 2WD'))
 .select('configuration', 'model', 'VClass', 'fuelType', 'city08',
     'highway08'))
trucks_2019 = (autos
 .with_row_index('configuration')
 .filter((pl.col('year') == 2019) 
        & pl.col('VClass').cast(pl.Utf8).str.contains('Pickup'))
 .select('configuration', 'make', 'model', 'VClass',
         'fuelType', 'city08', 'highway08'))
print(trucks_ford_2018)

In [ ]:
print(trucks_2019)

In [ ]:
print(trucks_ford_2018
 .join_where(
     trucks_2019,
     pl.col('VClass') == pl.col('VClass_candidate'),
     pl.col('fuelType') == pl.col('fuelType_candidate'),
     pl.col('city08_candidate') >= pl.col('city08') + 3,
     pl.col('highway08_candidate') >= pl.col('highway08'),
     suffix='_candidate',
 )
 .with_columns(city_gain=pl.col('city08_candidate') - pl.col('city08'))
 .sort('configuration_candidate')
 .select(
         'make', 'model_candidate', 'city08', 'city08_candidate',
         'configuration', 'configuration_candidate',
         'city_gain', 'highway08_candidate'))

In [ ]:
print(trucks_ford_2018
 .join_where(
     trucks_2019,
     pl.col('VClass') == pl.col('VClass_candidate'),
     pl.col('fuelType') == pl.col('fuelType_candidate'),
     pl.col('city08_candidate') >= pl.col('city08') + 3,
     pl.col('highway08_candidate') >= pl.col('highway08'),
     how='full',
     suffix='_candidate',
 )
 .select('configuration', 'model',
         'configuration_candidate', 'model_candidate')
 .sort('configuration', 'configuration_candidate', nulls_last=True))

In [ ]:
print(models_2018
 .rename({'mpg_2018': 'city_mpg'})
 .join(models_2019
            .rename({'mpg_2019': 'city_mpg'}),
       on='model', how='left', suffix='_2019', validate='1:1')
 .sort('model'))

In [ ]:
print(models_2018.join(models_2019, on='model', how='full',
    coalesce=False)
 .sort('model', 'model_right', nulls_last=True))

In [ ]:
print(models_2018.join(models_2019, on='model', how='full', coalesce=True)
 .sort('model'))

In [ ]:
report_2017 = (autos
 .filter(pl.col('year') == 2017)
 .group_by('make')
 .agg(city_mpg=pl.col('city08').median(),
      highway_mpg=pl.col('highway08').median(),
      report_year=pl.col('year').first())
 .sort('make'))
print(report_2017)

In [ ]:
report_2018 = (autos
 .filter(pl.col('year') == 2018)
 .group_by('make')
 .agg(city_mpg=pl.col('city08').median(),
      highway_mpg=pl.col('highway08').median(),
      report_year=pl.col('year').first())
 .sort('make'))
print(report_2018)

In [ ]:
print(report_2017
 .update(report_2018, on='make', how='full', include_nulls=True)
 .sort('make'))

In [ ]:
report_2019 = (autos
 .filter(pl.col('year') == 2019)
 .group_by('make')
 .agg(city_mpg=pl.col('city08').median(),
      highway_mpg=pl.col('highway08').median(),
      report_year=pl.col('year').first())
 .sort('make'))
print(report_2019)

In [ ]:
print(report_2017
 .update(report_2018, on='make', how='full', include_nulls=True)
 .update(report_2019, on='make', how='full', include_nulls=True)
 .sort('make'))

In [ ]:
print(report_2017
 .update(report_2018, on='make', how='full', include_nulls=True)
 .update(report_2019, on='make', how='full', include_nulls=True)
 .filter(pl.col('report_year') < 2019)
 .sort('make'))

In [ ]:
print(report_2017
 .select('make', city_2017=pl.col('city_mpg'))
 .join(report_2018.select('make', city_2018=pl.col('city_mpg')),
       on='make', how='full', coalesce=True, validate='1:1')
 .join(report_2019.select('make', city_2019=pl.col('city_mpg')),
       on='make', how='full', coalesce=True, validate='1:1')
 .sort('make'))

In [ ]:
fuel_options = (pl.read_csv('data/vehicles.csv', use_pyarrow=True,
 columns=['year', 'make', 'fuelType', 'atvType', 'city08'])
 .filter(pl.col('year') == 2019))
print(fuel_options)

In [ ]:
gas_mpg = (fuel_options
 .filter(pl.col('fuelType').is_in(['Regular', 'Premium', 'Midgrade']) &
         (pl.col('atvType') == ''))
 .group_by('make')
 .agg(gas_city_mpg=pl.col('city08').median()))
print(gas_mpg.sort('make'))

In [ ]:
hybrid_mpg = (fuel_options
 .filter(pl.col('atvType') == 'Hybrid')
 .group_by('make')
 .agg(hybrid_city_mpg=pl.col('city08').median()))
print(hybrid_mpg.sort('make'))

In [ ]:
print(pl.concat([gas_mpg, hybrid_mpg], how='align_left'))

In [ ]:
print(pl.concat([gas_mpg, hybrid_mpg], how='align_inner'))

In [ ]:
aligned_mpg = pl.align_frames(gas_mpg, hybrid_mpg, on='make')
print(aligned_mpg[0])

In [ ]:
print(aligned_mpg[1])

In [ ]:
print(pl.concat([
    (autos.group_by('make').len(name='make_records')
     .sort(['make_records', 'make'], descending=[True, False]).slice(0,
     5)),
    (autos.group_by('fuelType').len(name='fuel_records')
     .sort(['fuel_records', 'fuelType'], descending=[True,
     False]).slice(0, 3)),
], how='horizontal_extend'))

# String Manipulation

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
from highlight_text import fig_text
from polars.testing import assert_frame_equal

In [ ]:
_ = pl.Config.set_tbl_width_chars(73)
_ = pl.Config.set_float_precision(2)
_ = pl.Config.set_tbl_cols(6)
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_formatting('UTF8_BORDERS_ONLY')
_ = pl.Config.set_tbl_hide_dtype_separator(True)

In [ ]:
raw = pl.read_csv('data/__mharrison__2020-2021.csv',
    schema_overrides={'Tweet id': pl.String})
print(raw)

In [ ]:
input_cols = ['Tweet id', 'Tweet permalink', 'Tweet text', 'time',
              'impressions', 'engagements', 'engagement rate',
              'retweets', 'replies', 'likes', 'user profile clicks']
print(raw.select(input_cols))

In [ ]:
print(raw  # Capture
 .select(input_cols)  # Limit
 .rename(lambda name: name.lower().replace(' ', '_'))  )  # Edit

In [ ]:
print(raw  # Capture
 .select(input_cols)  # Limit
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit
 .schema)

In [ ]:
print(raw  # Capture
 .select(input_cols)  # Limit
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit
 .select('time', parsed_time=pl.col('time').str.to_datetime(
     '%Y-%m-%d %H:%M:%S%z', time_zone='UTC')))

In [ ]:
count_cols = ['impressions', 'engagements', 'retweets',
              'replies', 'likes', 'user_profile_clicks']
print(raw  # Capture
 .select(input_cols)  # Limit
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit
 .select(pl.col(count_cols).min()))

In [ ]:
print(raw  # Capture
 .select(input_cols)  # Limit
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit
 .select(pl.col(count_cols).max()))

In [ ]:
print(raw  # Capture
 .select(input_cols)  # Limit
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit
 .select((pl.col(count_cols) % 1 != 0).sum()))

In [ ]:
print(raw                   # Capture
 .select(input_cols)       # Limit to input columns
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit column names
 # Audit column types
 .with_columns(
     pl.col("time").str.to_datetime(
         "%Y-%m-%d %H:%M:%S%z", time_zone="UTC"),
     pl.col("impressions").cast(pl.UInt32),
     pl.col("likes").cast(pl.UInt32),
     pl.col("retweets").cast(pl.UInt32),
     pl.col("replies").cast(pl.UInt32),
     pl.col("engagements").cast(pl.UInt32),
     pl.col("user_profile_clicks").cast(pl.UInt32),
     pl.col("engagement_rate").cast(pl.Float64)
 )
)

In [ ]:
print(raw                   # Capture
 .select(input_cols)       # Limit to input columns
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit column names
 # Audit column types
 .with_columns(
     pl.col("time").str.to_datetime(
         "%Y-%m-%d %H:%M:%S%z", time_zone="UTC"),
     pl.col("impressions").cast(pl.UInt32),
     pl.col("likes").cast(pl.UInt32),
     pl.col("retweets").cast(pl.UInt32),
     pl.col("replies").cast(pl.UInt32),
     pl.col("engagements").cast(pl.UInt32),
     pl.col("user_profile_clicks").cast(pl.UInt32),
     pl.col("engagement_rate").cast(pl.Float64)
 )
 .null_count()  # Check for missing values
)

In [ ]:
null_counts = (raw                   # Capture
 .select(input_cols)       # Limit to input columns
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit column names
 # Audit column types
 .with_columns(
     pl.col("time").str.to_datetime(
         "%Y-%m-%d %H:%M:%S%z", time_zone="UTC"),
     pl.col("impressions").cast(pl.UInt32),
     pl.col("likes").cast(pl.UInt32),
     pl.col("retweets").cast(pl.UInt32),
     pl.col("replies").cast(pl.UInt32),
     pl.col("engagements").cast(pl.UInt32),
     pl.col("user_profile_clicks").cast(pl.UInt32),
     pl.col("engagement_rate").cast(pl.Float64)
 )
 .null_count()  # Check for missing values
)

null_counts = null_counts.select(
    name
    for name, count in null_counts.row(0, named=True).items()
    if count > 0
)
print(null_counts)

In [ ]:
print(raw  # Capture
 .select(input_cols)  # Limit
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit
 .select(pl.col(count_cols + ['engagement_rate']).is_nan().sum()))

In [ ]:
print(raw                   # Capture
 .select(input_cols)       # Limit to input columns
 .rename(lambda name: name.lower().replace(' ', '_'))  # Edit column names
 # Audit column types
 .with_columns(
     pl.col("time").str.to_datetime(
         "%Y-%m-%d %H:%M:%S%z", time_zone="UTC"),
     pl.col("impressions").cast(pl.UInt32),
     pl.col("likes").cast(pl.UInt32),
     pl.col("retweets").cast(pl.UInt32),
     pl.col("replies").cast(pl.UInt32),
     pl.col("engagements").cast(pl.UInt32),
     pl.col("user_profile_clicks").cast(pl.UInt32),
     pl.col("engagement_rate").cast(pl.Float64)
 )
 # Null - no missing
 # Engineer new columns
 .with_columns(
    engagement_pct=pl.col('engagement_rate') * 100))

In [ ]:
def tweak_twitter(df):
    cols = ['Tweet id', 'Tweet permalink', 'Tweet text', 'time',
            'impressions', 'engagements', 'engagement rate',
            'retweets', 'replies', 'likes', 'user profile clicks']
    final_cols = ['tweet_id', 'tweet_permalink', 'tweet_text', 'time',
                  'impressions', 'engagements', 'engagement_rate',
                  'retweets', 'replies', 'likes', 'user_profile_clicks',
                  'engagement_pct']
    return (df                 # Capture
      .select(cols)       # Limit to input columns
      # Edit column names
      .rename(lambda name: name.lower().replace(' ', '_'))
      # Audit column types
      .with_columns(
          pl.col("time").str.to_datetime(
              "%Y-%m-%d %H:%M:%S%z", time_zone="UTC"),
          pl.col("impressions").cast(pl.UInt32),
          pl.col("likes").cast(pl.UInt32),
          pl.col("retweets").cast(pl.UInt32),
          pl.col("replies").cast(pl.UInt32),
          pl.col("engagements").cast(pl.UInt32),
          pl.col("user_profile_clicks").cast(pl.UInt32),
          pl.col("engagement_rate").cast(pl.Float64)
      )
      # Null - no missing
      # Engineer new columns
      .with_columns(
         engagement_pct=pl.col('engagement_rate') * 100)
      # Restrict output columns
      .select(final_cols)
    )

twit = tweak_twitter(raw)
print(twit)

In [ ]:
col = pl.col('tweet_permalink')
print([m for m in dir(col.str)
    if not m.startswith('_')])

In [ ]:
print(sorted([m for m in 
      set(dir(col.str)) & set(dir(''))
      if not m.startswith('_')]))

In [ ]:
'https://metasnake.com'.startswith('https://twitter.com')

In [ ]:
print(twit.filter(col.is_null() | ~col.str
    .starts_with('https://twitter.com/')))

In [ ]:
print('https://metasnake.com/effective-polars'.split('/'))

In [ ]:
print(twit
 .select(col.str.split('/'))
)

In [ ]:
print(twit
 .with_columns(username=col.str.split('/')
    .list.get(3))
)

In [ ]:
print(twit
 .select(col.str.split('/').list.join('/'))
)

In [ ]:
print([m for m in dir(col.list)
    if not m.startswith('_')])

In [ ]:
print(twit
 .select(col.str.split('/')
    .list.len())
)

In [ ]:
print(twit
 .select(col.str.split('/')
    .list.to_struct(['scheme', 'empty', 'host', 'username', 'resource', 
                     'status_id']))
)

In [ ]:
print(twit
 .select(col.str.split('/')
    .list.to_struct(['scheme', 'empty', 'host', 'username', 
                     'resource', 'status_id']))
 .unnest('tweet_permalink')
)

In [ ]:
url_fields = ['scheme', 'empty', 'host', 'username', 'resource',
    'status_id']
print(twit
 # this is a struct
 .select(url=col.str.split('/').list.to_struct(url_fields))
 # this is a list of strings
 .select(parts=pl.concat_list([pl.col('url').struct
     .field(name) for name in url_fields]))
)

In [ ]:
regex = r'^https://twitter\.com/([a-zA-Z0-9_]+)/status/([0-9]+)$'
print(twit.select(user=col.str.extract(regex, group_index=1)))

In [ ]:
print(twit
 .select('tweet_id',
    url_parts=pl.col('tweet_permalink').str.extract_groups(
        (
            '^https://twitter\\.com/(?<username>[a-zA-Z0-9_]+)'
            '/status/(?<status_id>[0-9]+)$'
        )))
 .unnest('url_parts'))

In [ ]:
tweet_col = pl.col('tweet_text')
print(twit
 .select(tweet_col.str.extract_all(r'\S+'))
 .select(pl.col('tweet_text').list.slice(0,5)) 
)

In [ ]:
print(twit
 .select(tweet_col.str.extract_all(r'\S+').list.len())
)

In [ ]:
print(twit
 .with_columns(token_count=tweet_col.str.extract_all(r'\S+').list.len())
)

In [ ]:
print(twit
 .select(tweet_col.str.extract_all(r'\S+'))
)

In [ ]:
print(twit
 .select(tweet_col.str.extract_all(r'\S+').list.eval(
    pl.element().str.starts_with('@')))
)

In [ ]:
print(twit
 .select(tweet_col.str.extract_all(r'\S+').str.starts_with('@'))
)

In [ ]:
print(twit
 .select(num_mentions=tweet_col.str.extract_all(r'\S+')
         .list.eval(pl.element().str.starts_with('@'))
         .list.sum())
)

In [ ]:
print(twit
 .with_columns(token_count=tweet_col.str.extract_all(r'\S+').list.len(),
     num_mentions=tweet_col.str.extract_all(r'\S+')
                   .list.eval(pl.element().str.starts_with('@'))
                   .list.sum())
)

In [ ]:
non_ascii = r'[^\x00-\x7F]'
tweet_col = pl.col('tweet_text')
print(twit
 .select(tweet_col,
     has_non_ascii=tweet_col.str.contains(non_ascii),
     non_ascii_text=tweet_col.str.extract_all(non_ascii).list.join(''))
)

In [ ]:
print(twit.select('time', 'engagements', 
   tweet_col, starts_with_mention=tweet_col.str.starts_with('@')))

In [ ]:
weekly = (twit
   .select(pl.col('time'), 'engagements',
        starts_with_mention=tweet_col.str.starts_with('@'))
 .sort('time')
 .group_by_dynamic('time', every='1w', group_by='starts_with_mention')
 .agg(mean_engagements=pl.col('engagements').mean())
 .pivot(index='time', on='starts_with_mention', values='mean_engagements')
 .rename({'false': 'Other posts', 'true': 'Leading mention'})
 .sort('time')
)
fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
for label, color in [('Other posts', '#42678a'), ('Leading mention',
    '#a0a0a0')]:
    _ = ax.plot(weekly['time'].to_list(), weekly[label].to_numpy(),
    label=label, color=color)
_ = ax.set(xlabel='Week (UTC)', ylabel='Mean engagements per post')
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
ax.spines[['top', 'left']].set_visible(False)
_ = ax.legend(frameon=False, fontsize=7)
fig.subplots_adjust(top=0.80, bottom=0.25, right=0.85)
fig.autofmt_xdate()
_ = fig_text(0.12, 0.94,
    'Weekly means summarize engagement by text group', fontsize=10,
    fig=fig)
_ = fig_text(0.12, 0.03,
    'Source: frozen 2020–2021 Twitter export; mean per observed post.',
    fontsize=6, fig=fig)
plt.close(fig)

In [ ]:
print(twit
 .filter(pl.col('tweet_text').str.to_lowercase().str
     .contains_any(['python', 'pandas']))
 .select('tweet_text'))

In [ ]:
print(twit
 .filter(pl.col('tweet_text').str.contains_any(['&amp;', '&lt;', '&gt;']))
 .select('tweet_text', cleaned=pl.col('tweet_text').str.replace_many(
     {'&amp;': '&', '&lt;': '<', '&gt;': '>'})))

In [ ]:
topics = ['Python', 'pandas', 'scikit-learn']
topic_matches = (twit
 .filter(pl.col('tweet_text').str.contains_any(
     topics, ascii_case_insensitive=True))
 .select('tweet_id', 'tweet_text',
     matches=pl.col('tweet_text').str.extract_many(
         topics, ascii_case_insensitive=True),
     byte_offsets=pl.col('tweet_text').str.find_many(
         topics, ascii_case_insensitive=True)))

In [ ]:
print(topic_matches
 .explode('matches', 'byte_offsets')
 .select('tweet_text', 'byte_offsets',
     string_slice=pl.col('tweet_text')
         .str.slice(pl.col('byte_offsets'), 20),
     byte_slice=pl.col('tweet_text')
         .cast(pl.Binary)
         .bin.slice(pl.col('byte_offsets'))
         .cast(pl.String)
         .str.slice(0, 20)))

In [ ]:
parsed_ids = (twit
 .select('tweet_id', 'tweet_permalink',
     status_text=pl.col('tweet_permalink').str.extract(
         r'^https://twitter\.com/[A-Za-z0-9_]+/status/([0-9]+)$'))
 .with_columns(
     status_number=pl.col('status_text').str.to_integer(dtype=pl.UInt64,
     strict=False),
     exported_number=pl.col('tweet_id').str.to_integer(dtype=pl.UInt64,
     strict=False))
 .with_columns(
     parse_failed=pl.col('status_number').is_null()
     | pl.col('exported_number').is_null(),
     ids_match=pl.col('status_number') == pl.col('exported_number')))
print(parsed_ids.filter(pl.col('parse_failed') | ~pl.col('ids_match')))

In [ ]:
print(parsed_ids.select('status_text',
    too_small=pl.col('status_text').str.to_integer(dtype=pl.UInt8,
       strict=False)))

In [ ]:
print(twit
 .filter(pl.col('tweet_text').str.contains(r'\$[0-9]'))
 .select('tweet_id', 'tweet_text',
     amount_text=pl.col('tweet_text').str
     .extract_all(r'\$([0-9]+(?:\.[0-9]{1,2})?)'))
 .explode('amount_text')
 .with_columns(amount=pl.col('amount_text').str.strip_prefix('$')
     .cast(pl.Decimal(12, 2))))

In [ ]:
tweet_payloads = (twit
 .select(record=pl.struct(
     tweet_id=pl.col('tweet_id'),
     time=pl.col('time'),
     permalink=pl.col('tweet_permalink'),
     engagements=pl.col('engagements')))
 .select(display=pl.col('record').cast(pl.String, strict=False),
         payload=pl.col('record').struct.json_encode()))
print(tweet_payloads)

In [ ]:
import json

print(json.dumps(json.loads(tweet_payloads['payload'][0]), indent=2))

In [ ]:
tweet_schema = pl.Struct({
    'tweet_id': pl.String, 'time': pl.Datetime('us', 'UTC'),
    'permalink': pl.String,
    'engagements': pl.UInt32,
})

In [ ]:
decoded_tweets = (tweet_payloads
 .select(pl.col('payload').str.json_decode(dtype=tweet_schema))
 .unnest('payload'))
print(decoded_tweets)
assert_frame_equal(
    decoded_tweets,
    twit.select('tweet_id', 'time', 'tweet_permalink', 'engagements')
        .rename({'tweet_permalink': 'permalink'}))

In [ ]:
question_posts = (twit
 .filter(pl.col('tweet_text').str.contains('?', literal=True))
 .select('tweet_id', 'tweet_text', 'engagements',
     topic=pl.col('tweet_text').str.to_lowercase()
         .str.extract_all(r'\b(?:python|pandas|scikit-learn)\b')
         .list.unique())
 .explode('topic')
 .filter(pl.col('topic').is_not_null())
 .sort('engagements', 'tweet_id', descending=[True, False])
 .group_by('topic', maintain_order=True).head(3)
 .sort('topic', 'engagements', descending=[False, True]))
print(question_posts.select('topic', 'tweet_text', 'engagements'))

In [ ]:
top_questions = (question_posts
 .group_by('topic', maintain_order=True)
 .agg(questions=pl.format('{}. {}',
     pl.int_range(1, pl.len() + 1), 'tweet_text').str.join('\n'))
)
print(top_questions)

In [ ]:
print(top_questions
 .select(
    pl.format('{} questions\n{}', 'topic', 'questions')
      .str.join('\n\n')).item())

In [ ]:
print(twit
 .select('tweet_id', character=pl.col('tweet_text').str.split(''))
 .explode('character')
 )

In [ ]:
print(twit
 .select('tweet_id', character=pl.col('tweet_text').str.split(''))
 .explode('character')
 .filter(pl.col('character').str.contains(r'[^\x00-\x7F]'))
 .group_by('character')
 .len(name='occurrences')
 .sort('occurrences', 'character', descending=[True, False]))

In [ ]:
print(twit
 .select(hashtag=pl.col('tweet_text').str.extract_all(r'#[A-Za-z0-9_]+'))
 .explode('hashtag')
 .group_by('hashtag').len(name='occurrences')
 .with_columns(pl.col('hashtag').cast(pl.Categorical))
 .sort('occurrences', 'hashtag', descending=[True, False])
 .with_columns(label=pl.col('hashtag').cat.slice(1)))

# Aggregation with Polars

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
from highlight_text import fig_text
from polars.testing import assert_frame_equal

In [ ]:
_ = pl.Config.set_tbl_width_chars(73)
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_cols(7)
_ = pl.Config.set_float_precision(2)

In [ ]:
from typing import TypeVar
Frame = TypeVar("Frame", pl.DataFrame, pl.LazyFrame)

def tweak_auto_cleaner(df: Frame) -> Frame:
    cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany',
            'drive', 'VClass', 'fuelType', 'barrels08', 'city08',
            'highway08', 'createdOn']
    return (df  # Capture: start with the original data
        .select(pl.col(cols))  # Limit: select input columns
        # Edit: keep the original column names
        .with_columns(  # Audit: apply the chosen types
            pl.col('year').cast(pl.Int16),
            pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
            pl.col(['displ', 'barrels08']).cast(pl.Float32),
            pl.col(['make', 'model', 'VClass', 'drive', 
                'fuelType']).cast(pl.Categorical),
            pl.col('createdOn')  # Audit: convert the text to a Date
              .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
              .str.to_date('%b %d %Y'),
        )  # Null: keep missing values and empty strings
        .with_columns(  # Engineer: extract transmission features
            is_automatic=pl.col('trany').str.contains('Automatic'),
            num_gears=pl.col('trany').str.extract(r'(\d+)').cast(pl.UInt8),
        )
        # Restrict: choose output columns
        .select(cols + ['is_automatic', 'num_gears'])  
    )

In [ ]:
raw = pl.read_csv('data/vehicles.csv', use_pyarrow=True)
autos = tweak_auto_cleaner(raw)
print(autos)

In [ ]:
print(autos
 .select(
    mean_city=pl.col('city08').mean(),
    std_city=pl.col('city08').std(ddof=1),
    var_city=pl.col('city08').var(ddof=1),
    q99_city=pl.col('city08').quantile(.99, interpolation='nearest')))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018) & (pl.col('make') == 'Mazda'))
 .select('displ')
 .drop_nulls()
 .unique()
 .sort('displ')
 .select(engine_sizes=pl.col('displ').implode(),
         median_observed=pl.col('displ').quantile(.5,
         interpolation='equiprobable'),
         median_interpolated=pl.col('displ').quantile(.5,
         interpolation='linear')))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018) & pl.col('fuelType').is_in(['Regular',
     'Premium']))
 .select(displacement_quartiles=pl.col('displ').quantile([.25, .5, .75])))

In [ ]:
print(autos
 .select('make', 'city08')
 .with_columns(
    mean_city=pl.col('city08').mean(),))

In [ ]:
print(autos
 .filter(pl.col('year') == 2018)
 .select(rows=pl.len(),
         known=pl.col('cylinders').count(),
         distinct=pl.col('cylinders').n_unique(),
         known_distinct=pl.col('cylinders').drop_nulls().n_unique()))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018) & pl.col('make').is_in(['Ford',
     'Tesla']))
 .with_columns(displ=pl.col('displ').fill_null(float('nan')))
 .select(minimum_liters=pl.col('displ').min(),
         minimum_with_nan=pl.col('displ').nan_min(),
         nan_count=pl.col('displ').is_nan().sum()))

In [ ]:
by_make = (autos
 .group_by('make')
 .agg(mean_city=pl.col('city08').mean(), records=pl.len())
 .sort('make'))
print(by_make)

In [ ]:
print(autos
 .group_by('make', maintain_order=True)
 .agg(mean_city=pl.col('city08').mean()))

In [ ]:
print(autos
 .group_by()
 .agg(observations=pl.col('city08'),
       total=pl.col('city08').sum(),
       rows=pl.len()))

In [ ]:
print(autos
 .group_by()
 .having(pl.col('city08').count() >= 2)
 .agg(rows=pl.len(),
       known_city=pl.col('city08').count(),
       mean_city=pl.col('city08').mean(),
       std_city=pl.col('city08').std(ddof=1)))

In [ ]:
ford_2018 = autos.filter((pl.col('year') == 2018)
    & (pl.col('make') == 'Ford'))
transmission_partitions = ford_2018.partition_by('is_automatic',
    as_dict=True)
print(transmission_partitions[(True,)])

In [ ]:
shortlists = (autos
 .filter((pl.col('year') == 2018)
         & pl.col('make').is_in(['Chevrolet', 'Ford', 'Honda'])
         & pl.col('trany').str.starts_with('Manual'))
 .group_by('make')
 .agg(shortlist=pl.struct('model', 'trany', 'displ', 'highway08')
      .sort_by(['highway08', 'model', 'trany', 'displ'],
               descending=[True, False, False, False])
      .head(3))
 .sort('make'))
print(shortlists)

In [ ]:
print(shortlists
 .explode('shortlist')
 .unnest('shortlist'))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018)
         & pl.col('make').is_in(['Audi', 'Chevrolet', 'Ford', 'Honda'])
         & pl.col('trany').str.starts_with('Manual'))
 .group_by('make')
 .len(name='eligible')
 .sort('make'))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018)
         & pl.col('make').is_in(['Audi', 'Chevrolet', 'Ford', 'Honda'])
         & pl.col('trany').str.starts_with('Manual'))
 .group_by('make')
 .having(pl.len() >= 3)
 .agg(shortlist=pl.struct('model', 'trany', 'displ', 'highway08')
      .sort_by(['highway08', 'model', 'trany', 'displ'],
               descending=[True, False, False, False])
      .head(3))
 .sort('make')
 .explode('shortlist')
 .unnest('shortlist'))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018)
         & pl.col('fuelType').is_in(['Regular', 'Premium']))
 .select('VClass', 'displ', 'city08', 'highway08')
 .group_by('VClass')
 .agg(cs.numeric().mean())
 .sort('VClass'))

In [ ]:
annual_fuels = (autos
 .filter(pl.col('year').is_between(2017, 2019)
         & pl.col('make').is_in(['Chevrolet', 'Ford', 'Toyota']))
 .group_by('make', 'year')
 .agg(fuels=pl.col('fuelType').unique().sort())
 .sort('make', 'year'))
print(annual_fuels)

In [ ]:
print(annual_fuels
 .group_by('make')
 .agg(years=pl.col('year'), annual_lists=pl.col('fuels'))
 .sort('make'))

In [ ]:
print(annual_fuels
 .group_by('make')
 .agg(all_fuels=pl.col('fuels').explode().unique().sort())
 .sort('make'))

In [ ]:
print(annual_fuels
 .group_by('make')
 .agg(all_fuels=pl.col('fuels').explode().unique().sort())
 .sort('make')
 .explode('all_fuels'))

In [ ]:
print(annual_fuels
 .with_columns(fuel_choices=pl.col('fuels').list.len(),
               offers_electric=pl.col('fuels').list
               .contains('Electricity')))

In [ ]:
print(annual_fuels
 .with_columns(electric_fuels=pl.col('fuels').list.filter(
     pl.element() == 'Electricity'))
 .with_columns(first_electric=pl.col('electric_fuels').list.get(0,
     null_on_oob=True)))

In [ ]:
print(annual_fuels
 .with_columns(gasoline_choices=pl.col('fuels').list.agg(
     pl.element().is_in(['Regular', 'Premium']).sum())))

In [ ]:
print(autos
 .with_row_index('original_row')
 .filter((pl.col('year') == 2018) & (pl.col('make') == 'Ford'))
 .select(
     first_row_ford=pl.col('trany').arg_unique(),
     first_row_original=pl.col('original_row')
         .gather(pl.col('trany').arg_unique()),
     trany=pl.col('trany').gather(pl.col('trany').arg_unique()),
     count=pl.col('trany').unique_counts(),
 ))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018) & (pl.col('make') == 'Ford'))
 .select(pl.col('trany').value_counts(name='records'))
 .unnest('trany')
 .sort(['records', 'trany'], descending=[True, False]))

In [ ]:
print(autos
 .filter(pl.col('year') == 2018)
 .group_by('make')
 .agg(records=pl.len(),
      cylinder_choices=pl.col('cylinders').drop_nulls().n_unique(),
      missing_cylinders=pl.col('cylinders').null_count())
 .sort('make'))

In [ ]:
print(autos
 .filter(pl.col('year') == 2018)
 .group_by('fuelType')
 .agg(common_transmissions=pl.col('trany').drop_nulls().mode().sort())
 .with_columns(common_transmission_count=pl.col('common_transmissions')
     .list.len())
 .sort(['common_transmission_count', 'fuelType'],
       descending=[True, False]))

In [ ]:
print(autos
 .select(
 exact_models=pl.col('model').n_unique(),
 estimated_models=pl.col('model').approx_n_unique()))

In [ ]:
print(autos
 .group_by('make', 'year')
 .agg(mean_city=pl.col('city08').mean(), records=pl.len())
 .sort('make', 'year'))

In [ ]:
print(autos
 .group_by('make')
 .agg(
        pl.col('city08', 'highway08').mean().name.suffix('_mean'),
        pl.col('city08', 'highway08').median().name.suffix('_median'),
    )
 .sort('make'))

In [ ]:
print(autos
 .group_by(decade=pl.col('year') // 10 * 10)
 .agg(first_year=pl.col('year').min(),
      last_year=pl.col('year').max(),
      records=pl.len(),
      median_city=pl.col('city08').median(),
      median_highway=pl.col('highway08').median())
 .filter(pl.col('decade') >= 2000)
 .sort('decade'))

In [ ]:
print(autos
 .filter(pl.col('city08').is_not_null())
 .group_by('make')
 .having(pl.len() >= 100)
 .agg(
     mean_city=pl.col('city08').mean(),
     best_city=pl.col('city08').max(),
     example_model=pl.col('model').max_by('city08'),
 )
 .sort('make'))

In [ ]:
print(autos
 .filter(pl.col('year') == 2018)
 .group_by('make')
 .agg(missing=pl.col('is_automatic').null_count(),
      any_automatic=pl.col('is_automatic').any(ignore_nulls=False),
      all_automatic=pl.col('is_automatic').all(ignore_nulls=False))
 .sort('make'))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018) & pl.col('make').is_in(['Ford',
     'Tesla']))
 .group_by('make')
 .agg(no_rows=pl.col('displ').is_empty(),
      no_known_engines=pl.col('displ').is_empty(ignore_nulls=True),
      no_large_engines=pl.col('displ').filter(pl.col('displ') > 4)
      .is_empty())
 .sort('make'))

In [ ]:
print(autos
 .with_columns(city_band=pl.col('city08').bin_intervals(
     [20, 30], labels=['under 20', '20 to under 30', '30 or more']))
 .group_by('city_band')
 .len()
 .sort('city_band'))

In [ ]:
print(autos
 .with_columns(
     quantile_band=pl.col('city08').bin_quantiles(4, labels=False),
     rank_band=pl.col('city08').bin_ranks(4, labels=False),
 )
 .group_by('city08')
 .agg(
     quantile_bands=pl.col('quantile_band').n_unique(),
     rank_bands=pl.col('rank_band').n_unique(),
 )
 .filter(pl.col('rank_bands') > 1)
 .sort('city08'))

In [ ]:
print(autos
 .select(pl.col('city08').hist(bins=[0, 20, 30, 150], include_category=True)))

In [ ]:
print(autos
 .select(pl.col('city08').hist(bins=[0, 20, 30, 150], include_category=True))
 .unnest('city08'))

In [ ]:
print(autos
 .select(pl.col('city08').hist(bin_count=4, include_category=True))
 .unnest('city08'))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018) & (pl.col('fuelType') == 'Regular'))
 .select('make', 'model', city_gallons=100 / pl.col('city08'),
         highway_gallons=200 / pl.col('highway08'))
 .with_columns(gallons_with_reserve=pl.sum_horizontal(
     'city_gallons', 'highway_gallons', ignore_nulls=False) + 1))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018) & (pl.col('fuelType') == 'Regular'))
 .select('make', 'model', city_gallons=100 / pl.col('city08'),
         highway_gallons=200 / pl.col('highway08'))
 .with_columns(gallons_with_reserve=pl.fold(
     acc=pl.lit(1.),
     function=lambda total, value: total + value,
     exprs=['city_gallons', 'highway_gallons'],
     return_dtype=pl.Float64)))

In [ ]:
print(autos
 .filter((pl.col('year') == 2018) & (pl.col('fuelType') == 'Regular'))
 .select('make', 'model', city_gallons=100 / pl.col('city08'),
         highway_gallons=200 / pl.col('highway08'))
 .with_columns(gallons_with_reserve=pl.cum_fold(
     acc=pl.lit(1.),
     function=lambda total, value: total + value,
     exprs=['city_gallons', 'highway_gallons'],
     return_dtype=pl.Float64)))

In [ ]:
top_n = (autos
 .group_by('make')
 .agg(pl.col('city08').count())
 .sort('city08')
 .tail(5))

print(top_n)

In [ ]:
pivoted = (autos
 .filter(pl.col('make').is_in(top_n['make'].implode()))
 .pivot(on='make', index='year', values='city08', 
        aggregate_function='median')
 .sort('year'))

print(pivoted)

In [ ]:
pivoted_expr = (autos
 .filter(pl.col('make').is_in(top_n['make'].implode()))
 .pivot(on='make', index='year', values='city08',
        aggregate_function=pl.element().median())
 .sort('year'))

print(pivoted_expr.equals(pivoted))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for make, offset in [('Chevrolet', -7), ('Dodge', 0),
                     ('Toyota', 7), ('Ford', 0)]:
    color = '#c62828' if make == 'Ford' else '#888888'
    width = 2 if make == 'Ford' else 1
    ax.plot(pivoted['year'], pivoted[make], color=color,
            linewidth=width)
    year, mpg = pivoted.select('year', make).drop_nulls().row(-1)
    ax.annotate(make, xy=(year, mpg), xytext=(7, offset),
                textcoords='offset points', color=color,
                fontsize=9, va='center')
ax.set_title("Comparing Ford's city mileage over time", loc='left')
ax.set(xlabel='Model year', ylabel='Median city MPG / MPGe',
       xlim=(1983, 2025), ylim=(12, 31))
ax.set_xticks([1985, 1995, 2005, 2015, 2020])
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_color('#bbbbbb')
ax.tick_params(axis='both', length=0, labelsize=9)
ax.set_axisbelow(True)
ax.grid(axis='y', color='#eeeeee', linewidth=0.6)
fig.tight_layout()

In [ ]:
print(pivoted)

In [ ]:
print(pivoted
 .unpivot(index='year', on=['Chevrolet', 'Ford', 'GMC', 
                                   'Toyota', 'Dodge']))

In [ ]:
print(pivoted
 .unpivot(index='year', on=['Chevrolet', 'Ford', 'GMC', 
                            'Toyota', 'Dodge'],
          value_name='median_city_mpg', variable_name='make'))

In [ ]:
top3 = (autos
 .group_by('make')
 .len()
 .sort(['len', 'make'], descending=[True, False])
 .head(3))
print(top3)

In [ ]:
# First, group by year and make and create two aggregation columns.
year_ranges = (autos
 .join(top3.select('make'), on='make', how='semi')
 .group_by('year', 'make')
 .agg(min_city08=pl.col('city08').min(),
     max_city08=pl.col('city08').max())
 .sort('year', 'make'))
print(year_ranges)

In [ ]:
# Now pivot both aggregated columns into columns for each make.
wide_ranges = (year_ranges
 .pivot(
    on='make', index='year', values=['min_city08', 'max_city08'],
    sort_columns=True,
)
 .sort('year'))
print(wide_ranges)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
for make, color in zip(top3['make'].cast(pl.String), ['#245d83',
    '#777777', '#aaaaaa'], strict=True):
    values = (year_ranges
     .filter(pl.col('make') == make)
     .sort('year'))
    zorder = 3 if make == 'Chevrolet' else 2
    _ = ax.plot(values['year'], values['max_city08'], color=color,
    label=make, zorder=zorder)
    _ = ax.plot(values['year'], values['min_city08'], color=color,
    linestyle='--', zorder=zorder)
_ = ax.set_xlabel('Model year')
_ = ax.set_xticks([1985, 1995, 2005, 2015])
_ = ax.set_ylabel('City MPG / MPGe')
ax.yaxis.set_label_position('right')
ax.yaxis.tick_right()
ax.spines[['top', 'left']].set_visible(False)
_ = ax.legend(frameon=False, fontsize=8)
_ = fig_text(.12, .98, 'Reported mileage ranges vary by make and year',
    fontsize=11, va='top')
_ = fig_text(.12, .89,
    'Solid: maximum · Dashed: minimum · Three most common makes',
    fontsize=8)
_ = fig_text(.12, .04, (
    'Source: book EPA vehicle snapshot; configuration'
    's are not sales weighted'
), fontsize=7)
fig.subplots_adjust(top=.78, bottom=.18, right=.88)
plt.close(fig)

In [ ]:
engine_source = (pl.scan_csv('data/clean.csv')
 .pipe(tweak_auto_cleaner)
 .filter(pl.col('year').is_between(2000, 2005)))

lazy_engines = (engine_source
 .pivot(on='make', index='year',
        values='displ', aggregate_function='median')
 .sort('year'))

In [ ]:
lazy_engines = (engine_source
 .pivot(on='make', on_columns=['Ford', 'Toyota', 'Tesla'], index='year',
        values='displ', aggregate_function='median')
 .sort('year'))
print(dict(lazy_engines.collect_schema()))

In [ ]:
print(lazy_engines.collect())

In [ ]:
make_names = (engine_source
 .select('make')
 .unique()
 .sort('make')
 .collect()
 .get_column('make')
 .to_list())
print(pl.Series('make', make_names))

In [ ]:
all_engines = (engine_source
 .pivot(on='make', on_columns=make_names, index='year',
        values='displ', aggregate_function='median')
 .sort('year')
 .collect())

In [ ]:
windowed = (autos
 .with_columns(mean_city=pl.col('city08').mean().over('make')))
print(windowed
 .select('make', 'city08', 'mean_city'))

In [ ]:
joined = (autos
 .select('make', 'city08')
 .join(by_make.select('make', 'mean_city'), 
       on='make', how='left', validate='m:1', maintain_order='left'))
print(joined)
assert_frame_equal(joined,
    windowed.select('make', 'city08', 'mean_city'))

In [ ]:
print(autos
 .filter(pl.col('year') == 2018)
 .select('make', 'model', 'displ', engine_group=pl.struct('VClass',
     'fuelType'))
 .with_columns(group_median_displ=pl.col('displ').median()
     .over('engine_group')))

In [ ]:
rank_vehicles = autos.filter(pl.col('year').is_in([2018, 2019]))
print(rank_vehicles
 .group_by('year')
 .agg(records=pl.len(),
      known_displacements=pl.col('displ').count())
 .sort('year'))

In [ ]:
ranked_engines = (rank_vehicles
 .select('year', 'model', 'displ',
     displacement_rank=pl.col('displ').rank().over('year'),
     rank_fraction=(pl.col('displ').rank() / pl.col('displ').count())
         .over('year'),
     fraction_all_records=(pl.col('displ').rank() / pl.len())
         .over('year'))
 .sort('year', 'displ', 'model', nulls_last=True))
print(ranked_engines.drop_nulls(subset='displ'))

In [ ]:
print(autos
 .filter(pl.col('fuelType') == 'Electricity', 
         pl.col('make').is_in(['Tesla', 'Ford', 'Toyota']))
 .group_by('make', 'year')
 .len(name='electric_records')
 .with_columns(cumulative_electric_records=pl.col('electric_records')
               .cum_sum().over('make', order_by='year'))
 .sort('make', 'year'))

# Data Filtering and Selection

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import polars.selectors as cs
from sklearn import decomposition

In [ ]:
_ = pl.Config.set_tbl_width_chars(73)
_ = pl.Config.set_float_precision(2)
_ = pl.Config.set_tbl_cols(6)
#pl.Config.set_tbl_formatting('ASCII_BORDERS_ONLY_CONDENSED')
_ = pl.Config.set_tbl_formatting('UTF8_BORDERS_ONLY')
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_hide_dtype_separator(True)

In [ ]:
from typing import TypeVar
Frame = TypeVar("Frame", pl.DataFrame, pl.LazyFrame)

def tweak_auto_cleaner(df: Frame) -> Frame:
    cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany',
            'drive', 'VClass', 'fuelType', 'barrels08', 'city08',
            'highway08', 'createdOn']
    return (df  # Capture: start with the original data
        .select(pl.col(cols))  # Limit: select input columns
        # Edit: keep the original column names
        .with_columns(  # Audit: apply the chosen types
            pl.col('year').cast(pl.Int16),
            pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
            pl.col(['displ', 'barrels08']).cast(pl.Float32),
            pl.col(['make', 'model', 'VClass', 'drive', 
                'fuelType']).cast(pl.Categorical),
            pl.col('createdOn')  # Audit: convert the text to a Date
              .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
              .str.to_date('%b %d %Y'),
        )  # Null: keep missing values and empty strings
        .with_columns(  # Engineer: extract transmission features
            is_automatic=pl.col('trany').str.contains('Automatic'),
            num_gears=pl.col('trany').str.extract(r'(\d+)').cast(pl.UInt8),
        )
        # Restrict: choose output columns
        .select(cols + ['is_automatic', 'num_gears'])  
    )

In [ ]:
raw = pl.read_csv('data/vehicles.csv', use_pyarrow=True)
autos = tweak_auto_cleaner(raw)
print(autos)

In [ ]:
print(autos
 .filter(pl.col('city08') > 40)
 .select('year', 'make', 'model', 'city08'))

In [ ]:
print(autos
 .filter(pl.col('city08') > 40)
 .filter(pl.col('make') == 'Toyota')
 .select('year', 'make', 'model', 'city08'))

In [ ]:
print(autos
 .filter(pl.col('city08') > 40,
          pl.col('make') == 'Toyota')
 .select('year', 'make', 'model', 'city08'))

In [ ]:
print(autos
 .filter((pl.col('make') == 'Toyota') & (pl.col('city08') > 40))
 .select('year', 'make', 'model', 'city08'))

In [ ]:
print(autos
 .filter(pl.col('make') == 'Toyota' & pl.col('city08') > 40))

In [ ]:
print(autos
 .filter(pl.col('displ').is_between(1.8, 2.2))
 .select('displ')
 .drop_nulls()
 .unique()
 .sort('displ')
 .with_columns(within_tenth=pl.col('displ').is_close(2.0, 
                    abs_tol=0.1, rel_tol=0.0),
               within_five_percent=pl.col('displ').is_close(2.0, 
                    abs_tol=0.0, rel_tol=0.05)))

In [ ]:
print(autos
 .filter(pl.col('displ').is_close(2.0, abs_tol=0.1, rel_tol=0.0))
 .select('year', 'make', 'model', 'displ')
 .sort('year', 'make', 'model', 'displ'))

In [ ]:
from datetime import datetime
mar2018 = datetime(2018, 3, 1)
print(autos
 .filter(pl.col('createdOn') >= mar2018)
 .sort('createdOn'))

In [ ]:
print(autos
 .filter(pl.col('city08').is_between(40, 50))
 .sort('city08')
 .select(['year', 'make', 'model', 'VClass', 'city08']))

In [ ]:
mar2018 = datetime(2018, 3, 1)
print(autos
 .filter(pl.col('createdOn').is_between(
     pl.lit(mar2018).dt.offset_by('-5y'),
     mar2018))
 .sort('createdOn'))

In [ ]:
print(autos
 .filter((pl.col('make') == 'Ford') & (pl.col('year') == 2018))
 .group_by('fuelType')
 .len(name='records')
 .sort('fuelType'))

In [ ]:
print(autos
 .filter((pl.col('make') == 'Ford') & (pl.col('year') == 2018))
 .select(
     electric_median=pl.col('city08')
         .filter(pl.col('fuelType').cast(pl.String).str
         .contains('Electricity'))
         .median(),
     non_electric_median=pl.col('city08')
         .filter(~pl.col('fuelType').cast(pl.String).str
         .contains('Electricity'))
         .median()))

In [ ]:
print(autos
 .filter((pl.col('make') == 'Ford') & (pl.col('year') == 2018))
 .select(
     total_records=pl.len(),
     electric_records=pl.col('city08')
         .filter(pl.col('fuelType').cast(pl.String).str
         .contains('Electricity'))
         .len(),
     non_electric_records=pl.col('city08')
         .filter(~pl.col('fuelType').cast(pl.String).str
         .contains('Electricity'))
         .len()))

In [ ]:
print(raw
 .filter(pl.col('id') == 39245)
 .select('id', 'year', 'make', 'model', 'city08'))

In [ ]:
print(raw
 .select(city_mpg=pl.col('city08').filter(pl.col('id') == 39245).item()))

In [ ]:
print(raw
 .select(city_mpg=pl.col('city08').filter(pl.col('id') == 0)
         .item(allow_empty=True)))

In [ ]:
print(raw
 .filter((pl.col('year') == 2018)
         & (pl.col('make') == 'Ford')
         & (pl.col('model') == 'F150 Pickup 2WD'))
 .select('id', 'model', 'city08')
 .sort('id'))

In [ ]:
print(raw
 .select(city_mpg=pl.col('city08').filter(
     (pl.col('year') == 2018)
     & (pl.col('make') == 'Ford')
     & (pl.col('model') == 'F150 Pickup 2WD'))
     .item(allow_empty=True)))

In [ ]:
def standardize(col):
  return (col - col.mean()) / col.std()

print(autos
 .filter(standardize(pl.col('city08')) > 3)
 .select(['year', 'make', 'model', 'VClass', 'city08']))

In [ ]:
print(autos
 .filter(standardize(pl.col('city08')).over(['year', 'make']) > 3)
 .select(['year', 'make', 'model', 'VClass', 'city08']))

In [ ]:
print(autos
 .with_columns(
     city_z=standardize(pl.col('city08')).over(['year', 'make'])
 )
 .filter(pl.col('city_z').is_finite() & (pl.col('city_z') > 3))
 .select('year', 'make', 'model', 'VClass', 'city08', 'city_z')
)

In [ ]:
print(autos
 .with_columns(
     model_age=(pl.col('year').max() - pl.col('year').min())
                .over('model')))

In [ ]:
print(autos
 .with_columns(
    model_age=(pl.col('year').max() - pl.col('year').min())
               .over('model'))
 .filter(pl.col('model_age') > 10))

In [ ]:
print(autos
 .group_by('fuelType')
 .len()
 .sort('fuelType'))

In [ ]:
fuel_str = pl.col('fuelType').cast(pl.String)
simple = (pl.when(fuel_str.str.contains('Diesel')).then(pl.lit('Diesel'))
  .when(fuel_str.str.contains('CNG|natural gas')).then(pl.lit('CNG'))
  .when(fuel_str.str.contains('Electricity'))
      .then(pl.lit('electric'))
  .when(fuel_str.str.contains('Gasoline') | 
        fuel_str.is_in(['Regular', 'Premium', 'Midgrade']))
      .then(pl.lit('gas'))
  .otherwise(pl.lit(None, dtype=pl.String)))

print(autos
 .with_columns(simple.alias('fuelTypeSimple')))

In [ ]:
print(autos
 .filter(simple == 'CNG'))

In [ ]:
top_n = (pl.col('make')
     .value_counts(sort=True)
     .struct[0]
     .head(10)
)


top_expr = (pl.when(pl.col('make').is_in(top_n.implode()))
    .then(pl.col('make'))
    .otherwise(pl.lit('Other'))
)

print(autos
 .with_columns(simple_make=top_expr))

In [ ]:
def limit(col_name, n=10, default='other'):
    col = pl.col(col_name)    
    top_n = (col
     .value_counts(sort=True)
     .struct[0]
     .head(n)
    )          
    return (pl.when(col.is_in(top_n.implode()))
     .then(col)
     .otherwise(pl.lit(default))
    )

print(autos
 .with_columns(simple_make=limit('make')))

In [ ]:
yearly_mpg = (autos
 .filter(
     pl.col('year').is_between(2010, 2019),
     pl.col('fuelType') == 'Regular'
 )
 .pivot(on='make', index='year', values='highway08',
        aggregate_function='mean')
 .sort('year')
)
print(yearly_mpg)

In [ ]:
print(yearly_mpg
 .select(pl.exclude('year').mean())
)

In [ ]:
print(yearly_mpg
 .pipe(lambda df: df.select(
     'year',
     *[column.name
      for column in df.select(pl.exclude('year').mean())
      if column.item() is not None and column.item() >= 30]
 ))
)

In [ ]:
print(yearly_mpg
 .pipe(lambda df: df.select(
     'year',
     *[column.name
      for column in df.select(pl.exclude('year').mean())
      if column.item() is not None and column.item() >= 30]
 ))
 .pipe(lambda df: df.select(
     'year', *sorted(name for name in df.columns if name != 'year')
 ))
)

In [ ]:
print(autos
 .drop_nulls(subset=['cylinders', 'displ']))

In [ ]:
print(autos
 .filter(pl.col('num_gears', 'is_automatic').is_null()))

In [ ]:
print(autos
 .filter(pl.col('num_gears').is_null() & 
        pl.col('is_automatic').is_null()))

In [ ]:
print(autos
 .filter(pl.all_horizontal(
     pl.col('num_gears', 'is_automatic').is_null())))

In [ ]:
print(autos
 .filter(~pl.all_horizontal(
     pl.col('num_gears', 'is_automatic').is_null())))

In [ ]:
print(autos
 .with_columns(pl.col('cylinders').fill_null(0)))

In [ ]:
print(autos
 .select(cyl_mean=pl.col('cylinders').mean(),
        cyl_fill0_mean=pl.col('cylinders').fill_null(0).mean()))

In [ ]:
nan_rows = pl.DataFrame({
    "id": [0, 1, 2, 3],
    "reading": [float("nan"), 1.0, None, 2.0],
    "reference": [1.0, float("nan"), None, 2.0],
})
print(nan_rows)

In [ ]:
print(nan_rows
 .drop_nans())

In [ ]:
print(nan_rows
 .lazy()
 .drop_nans(subset="reading")
 .collect())

In [ ]:
print(nan_rows
 .with_columns(clean_reading=pl.col('reading').fill_nan(None))
 .with_columns(filled_reading=pl.col('clean_reading')
     .fill_null(strategy='mean'))
)

# Sorting and Ordering in Polars

In [ ]:
import polars as pl

In [ ]:
_ = pl.Config.set_tbl_width_chars(73)
_ = pl.Config.set_float_precision(2)
_ = pl.Config.set_tbl_cols(6)
#pl.Config.set_tbl_formatting('ASCII_BORDERS_ONLY_CONDENSED')
_ = pl.Config.set_tbl_formatting('UTF8_BORDERS_ONLY')
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_hide_dtype_separator(True)

In [ ]:
from typing import TypeVar
Frame = TypeVar("Frame", pl.DataFrame, pl.LazyFrame)

def tweak_auto_cleaner(df: Frame) -> Frame:
    cols = ['year', 'make', 'model', 'displ', 'cylinders', 'trany',
            'drive', 'VClass', 'fuelType', 'barrels08', 'city08',
            'highway08', 'createdOn']
    return (df  # Capture: start with the original data
        .select(pl.col(cols))  # Limit: select input columns
        # Edit: keep the original column names
        .with_columns(  # Audit: apply the chosen types
            pl.col('year').cast(pl.Int16),
            pl.col(['cylinders', 'highway08', 'city08']).cast(pl.UInt8),
            pl.col(['displ', 'barrels08']).cast(pl.Float32),
            pl.col(['make', 'model', 'VClass', 'drive', 
                'fuelType']).cast(pl.Categorical),
            pl.col('createdOn')  # Audit: convert the text to a Date
              .str.replace(r'^\w+ (\w+ \d+) .* (\d{4})$', '$1 $2')
              .str.to_date('%b %d %Y'),
        )  # Null: keep missing values and empty strings
        .with_columns(  # Engineer: extract transmission features
            is_automatic=pl.col('trany').str.contains('Automatic'),
            num_gears=pl.col('trany').str.extract(r'(\d+)').cast(pl.UInt8),
        )
        # Restrict: choose output columns
        .select(cols + ['is_automatic', 'num_gears'])  
    )

In [ ]:
raw = pl.read_csv('data/vehicles.csv', use_pyarrow=True)
autos = tweak_auto_cleaner(raw)

In [ ]:
print(autos
 .sort('year')
 .select('year', 'make', 'model')
)

In [ ]:
names = ['Al', 'Bob', 'Charlie', 'Dan', 'Edith', 'Frank']
print(sorted(names, key=len))

In [ ]:
print(autos
 .sort(by=pl.col('make').cast(pl.String).str.len_chars())
 .with_columns(make_length=pl.col('make').cast(pl.String)
     .str.len_chars())
 .select('year', 'make', 'model', 'make_length')
)

In [ ]:
print(autos
 .sort(pl.mean_horizontal(
     pl.col('city08', 'highway08').cast(pl.Float64)))
 .with_columns(mean_mpg=pl.mean_horizontal(
     pl.col('city08', 'highway08').cast(pl.Float64)))
 .select('make', 'model', 'city08', 'highway08', 'mean_mpg')
)

In [ ]:
print(autos
 .sort(pl.col('VClass'))
 .select('make', 'model', 'VClass')
)

In [ ]:
print(autos
 .sort(pl.col('city08').mean().over(pl.col('make')))
 .with_columns(make_avg=pl.col('city08').mean().over(pl.col('make')))
 .select('year', 'make', 'model', 'city08', 'make_avg')
)

In [ ]:
print(autos
 .sort(['year', 'make'])
)

In [ ]:
print(autos
 .sort(['year', 'make', 'model'])
)

In [ ]:
print(autos
 .sort(['year', 'make', 'model'], 
       descending=[True, False, False])
)

In [ ]:
print(autos
 .with_columns(month=pl.col('createdOn').dt.strftime('%B'))
 .sort('month')
)

In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November',
               'December']
print(autos
 .with_columns(month=pl.col('createdOn').dt.strftime('%B')
     .cast(pl.Enum(month_order)))
 .sort('month')
)

In [ ]:
print(autos
 .group_by(pl.col('make'))
 .len()
 .sort('make')
)

In [ ]:
print(autos
 .group_by(pl.col('make'), maintain_order=True)
 .len()
)

In [ ]:
print(autos
 .with_row_index('input_position')
 .filter(pl.col('year') == 2019)
 .sort('make', 'model')
 .filter((pl.col('make') == 'Chevrolet') & (pl.col('model') == 'Camaro'))
 .select('input_position', 'make', 'model', 'city08', 'highway08')
)

In [ ]:
print(autos
 .with_row_index('input_position')
 .filter(pl.col('year') == 2019)
 .sort('make', 'model', maintain_order=True)
 .filter((pl.col('make') == 'Chevrolet') & (pl.col('model') == 'Camaro'))
 .select('input_position', 'make', 'model', 'city08', 'highway08')
)

In [ ]:
print(autos
 .filter((pl.col('year') == 2019) & (pl.col('make') == 'Ford'))
 .select('model', 'trany', 'num_gears')
 .sort('num_gears', nulls_last=True, maintain_order=True)
)

In [ ]:
print(autos
 .filter((pl.col('year') == 2019) & (pl.col('make') == 'Ford'))
 .select('model', 'trany', 'num_gears')
 .sort('num_gears', descending=True, nulls_last=True, maintain_order=True)
)

In [ ]:
autos_shuffled = autos.sample(fraction=1, shuffle=True, seed=42)
autos_year_sorted = autos_shuffled.sort('year')

In [ ]:
print('Original:', autos['year'].flags)
print('Shuffled:', autos_shuffled['year'].flags)
print('Sorted:', autos_year_sorted['year'].flags)

In [ ]:
%%timeit 
autos.filter(pl.col('year') == 1994)

In [ ]:
%%timeit 
autos_shuffled.filter(pl.col('year') == 1994)

In [ ]:
%%timeit 
autos_year_sorted.filter(pl.col('year') == 1994)

In [ ]:
regular_2019 = (autos
 .filter((pl.col('year') == 2019) & (pl.col('fuelType') == 'Regular'))
 .select('make', 'model', 'displ', 'city08', 'highway08')
)
print(regular_2019
 .top_k(5, by=['highway08', 'displ'], reverse=[False, True])
 .sort(['highway08', 'displ'], descending=[True, False])
)

In [ ]:
print(regular_2019
 .sort('highway08', descending=True, nulls_last=True, maintain_order=True)
 .head(3)
)

In [ ]:
print(regular_2019
 .select(model=pl.col('model')
     .top_k_by(['highway08', 'displ'], 5, reverse=[False, True])
     .sort())
)

In [ ]:
print(autos
 .filter(pl.col('year') == 2019)
 .sort(['city08', 'highway08'], descending=[True, True],
       nulls_last=True, maintain_order=True)
 .unique(subset=['make', 'model'], keep='first', maintain_order=True)
 .select('make', 'model', 'displ', 'trany', 'city08', 'highway08'))

# Time Series Analysis

In [ ]:
import datetime
from zoneinfo import ZoneInfo, available_timezones

import matplotlib.pyplot as plt
import polars as pl
import holidays

In [ ]:
_ = pl.Config.set_tbl_width_chars(73)
_ = pl.Config.set_float_precision(2)
_ = pl.Config.set_tbl_cols(6)
#pl.Config.set_tbl_formatting('ASCII_BORDERS_ONLY_CONDENSED')
_ = pl.Config.set_tbl_formatting('UTF8_BORDERS_ONLY')
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_hide_dtype_separator(True)

In [ ]:
raw = pl.read_csv('data/devilclean.txt', separator='\t',
                  schema_overrides={'site_no': pl.String})
print(raw)

In [ ]:
print(raw  # Capture: start with the source frame
    .select(  # Limit: select required inputs
        'agency_cd', 'site_no', 'datetime', 'tz_cd', '144166_00060',
        '144167_00065'
    )
    # Edit: name measurements
    .rename({'144166_00060': 'cfs', '144167_00065': 'gage_height'})  
    .with_columns(  # Audit: enforce identifier and measurement types
        pl.col('site_no').cast(pl.String),
        pl.col('cfs', 'gage_height').cast(pl.Float64),
        pl.col('datetime').str.to_datetime('%Y-%m-%d %H:%M'),
    )
    # Null: retain missing measurements
    # Engineer: no additional features needed for the date-parsing
    # examples
    # Restrict: final columns
    .select('agency_cd', 'site_no', 'tz_cd', 'datetime', 'cfs',
    'gage_height')  
)

In [ ]:
def tweak_river(df_: pl.DataFrame) -> pl.DataFrame:
    return (df_  # Capture: start with the source frame
        .select(  # Limit: select required inputs
            'agency_cd', 'site_no', 'datetime', 'tz_cd', '144166_00060',  
            '144167_00065'
        )
        # Edit: name measurements
        .rename({'144166_00060': 'cfs', '144167_00065': 'gage_height'})  
        .with_columns(  # Audit: enforce identifier and measurement types
            pl.col('site_no').cast(pl.String),
            pl.col('cfs', 'gage_height').cast(pl.Float64),
            pl.col('datetime').str.to_datetime('%Y-%m-%d %H:%M'),
        )
        # Null: retain missing measurements
        # Engineer: no additional features needed for the date-parsing
        # examples
        # Restrict: final columns
        .select('agency_cd', 'site_no', 'tz_cd', 'datetime', 'cfs', 
                'gage_height')  
    )

dd = tweak_river(raw)
print(dd)

In [ ]:
print(pl.read_csv('data/devilclean.txt', separator='\t', 
                  try_parse_dates=True))

In [ ]:
format = '%Y-%m-%d %H:%M'
print(raw
.select(original=pl.col('datetime'),
        to_datetime=pl.col('datetime').str.to_datetime(format),
        to_date=pl.col('datetime').str.to_date(format),
        strptime=pl.col('datetime').str.strptime(pl.Datetime, format),
        # line below fails
        #cast=pl.col('datetime').cast(pl.Datetime)
        )
)

In [ ]:
print(dd
 .select(
     original=pl.col('datetime'),
     milliseconds=pl.col('datetime').dt.cast_time_unit('ms'),
 )
 .with_columns(
     back_to_microseconds=pl.col('milliseconds').dt.cast_time_unit('us')
 )
)

In [ ]:
print(raw
 .select(datetime=pl.col('datetime').str.to_datetime(
     '%Y-%m-%d %H:%M', time_unit='ms'
 ))
)

In [ ]:
print(tweak_river(raw)
['datetime']
.dtype.time_zone
)

In [ ]:
sorted(available_timezones())[:5]

In [ ]:
format = '%Y-%m-%d %H:%M'
print(raw
.select(original=pl.col('datetime'),
        naive=pl.col('datetime').str.to_datetime(format),
        # utc is wrong if the original time is local Mountain time
        utc=pl.col('datetime').str.to_datetime(format)
            .dt.replace_time_zone('UTC'), 
        # Denver is also wrong if we start from UTC
        Denver=pl.col('datetime').str.to_datetime(format)
            .dt.replace_time_zone('UTC')
            .dt.convert_time_zone('America/Denver'),
        Denver2=(pl.col('datetime') + ' ' + (pl.col('tz_cd')
            .str.replace('MST', '-0700').str.replace('MDT', '-0600')))
            .str.to_datetime('%Y-%m-%d %H:%M %z', time_zone='UTC')
            .dt.convert_time_zone('America/Denver'),
        Denver3=(pl.col('datetime').str.to_datetime(format,
                 time_zone='America/Denver', ambiguous='earliest'))
        )
)

In [ ]:
format = '%Y-%m-%d %H:%M'
print(raw
.select(
        Denver2=(pl.col('datetime') + ' ' + (pl.col('tz_cd')
            .str.replace('MST', '-0700').str.replace('MDT', '-0600')))
            .str.to_datetime('%Y-%m-%d %H:%M %z', time_zone='UTC')
            .dt.convert_time_zone('America/Denver'),            
        Denver3=(pl.col('datetime').str.to_datetime(format,
                 time_zone='America/Denver', ambiguous='earliest'))
        )
.filter(pl.col('Denver3') != pl.col('Denver2'))
)

In [ ]:
fall_readings = (raw
 .filter(pl.col('datetime').str.starts_with('2018-11-04'))
 .select(
     Denver2=(pl.col('datetime') + ' ' + pl.col('tz_cd')
         .str.replace('MST', '-0700').str.replace('MDT', '-0600'))
         .str.to_datetime('%Y-%m-%d %H:%M %z', time_zone='UTC')
         .dt.convert_time_zone('America/Denver'),
     Denver3=pl.col('datetime').str.to_datetime(
         '%Y-%m-%d %H:%M', time_zone='America/Denver',
         ambiguous='earliest'),
 )
 .sort('Denver2')
 .with_columns(
     gap2=pl.col('Denver2').diff().dt.total_minutes(),
     gap3=pl.col('Denver3').diff().dt.total_minutes(),
 )
)

In [ ]:
_ = pl.Config.set_tbl_rows(10)

In [ ]:
print(fall_readings
 .filter(pl.col('Denver2').dt.hour() == 1)
 .select(
     Denver2=pl.col('Denver2').dt.strftime('%H:%M %z'),
     Denver3=pl.col('Denver3').dt.strftime('%H:%M %z'),
     gap2_minutes=pl.col('gap2'),
     gap3_minutes=pl.col('gap3'),
 )
)

In [ ]:
_ = pl.Config.set_tbl_rows(6)

In [ ]:
print(raw  # Capture: start with the source frame
    .select(  # Limit: select required inputs
        'agency_cd', 'site_no', 'datetime', 'tz_cd', '144166_00060',
        '144167_00065'
    )
    # Edit: name measurements
    .rename({'144166_00060': 'cfs', '144167_00065': 'gage_height'})
    .with_columns(  # Audit: enforce identifier and measurement types
        pl.col('site_no').cast(pl.String),
        pl.col('cfs', 'gage_height').cast(pl.Float64),
    )
    # Null: retain missing measurements; missing does not mean zero
    .with_columns(  # Engineer: combine local time with its source offset
        datetime=(pl.col('datetime') + ' ' + pl.col('tz_cd')
            .str.replace('MST', '-0700').str.replace('MDT', '-0600'))
            .str.to_datetime('%Y-%m-%d %H:%M %z', time_zone='UTC')
            .dt.convert_time_zone('America/Denver')
    )
    # Restrict: final columns
    .select('agency_cd', 'site_no', 'cfs', 'gage_height', 'datetime')
)

In [ ]:
def tweak_river(
    df_: pl.DataFrame, cfs_col: str, gage_height_col: str
) -> pl.DataFrame:
    return (df_  # Capture: start with the source frame
        .select(  # Limit: select required inputs
            'agency_cd', 'site_no', 'datetime', 'tz_cd', cfs_col,
            gage_height_col
        )
        # Edit: name measurements
        .rename({cfs_col: 'cfs', gage_height_col: 'gage_height'})
        .with_columns(  # Audit: enforce identifier and measurement types
            pl.col('site_no').cast(pl.String),
            pl.col('cfs', 'gage_height').cast(pl.Float64),
        )
        # Null: retain missing measurements; missing does not mean zero
        # Engineer: combine local time with its source offset
        .with_columns(
            datetime=(pl.col('datetime') + ' ' + pl.col('tz_cd')
                .str.replace('MST', '-0700').str.replace('MDT', '-0600'))
                .str.to_datetime('%Y-%m-%d %H:%M %z', time_zone='UTC')
                .dt.convert_time_zone('America/Denver')
        )
        # Restrict: final columns
        .select('agency_cd', 'site_no', 'cfs', 'gage_height', 'datetime')
    )

dd = tweak_river(raw, cfs_col='144166_00060',
                 gage_height_col='144167_00065')
print(dd)

In [ ]:
print(dd
 .sort('datetime')
 .select(gap_minutes=pl.col('datetime').diff().dt.total_seconds() / 60)
 .select(
     mean_minutes=pl.col('gap_minutes').mean(),
     std_minutes=pl.col('gap_minutes').std(),
     variance_minutes_squared=pl.col('gap_minutes').var(),
 )
)

In [ ]:
print(dd
 .sort('datetime')
 .with_columns(previous_time=pl.col('datetime').shift(1),
               gap=pl.col('datetime').diff())
 .filter(pl.col('gap') > pl.duration(hours=12))
 .sort('gap', descending=True)
 .select('previous_time', 'datetime',
         iso_gap=pl.col('gap').dt.to_string(),
         compact_gap=pl.col('gap').dt.to_string('polars'))
)

In [ ]:
print(dd
 .with_columns(year=pl.col('datetime').dt.year())
 .group_by('year')
 .agg(pl.col(pl.Float64).mean())
 .sort('year')
 )

In [ ]:
print(dd
 .group_by_dynamic(index_column='datetime', every='1y')
 .agg(pl.col(pl.Float64).mean())
 )

In [ ]:
print(dd
 .sort('datetime')
 .group_by_dynamic(index_column='datetime', every='1y')
 .agg(pl.col(pl.Float64).mean())
 )

In [ ]:
print(dd
 .sort('datetime')  
 .group_by_dynamic(index_column='datetime', every='2mo')
 .agg(pl.col(pl.Float64).mean())
)

In [ ]:
print(dd
 .sort('datetime')  
 .group_by_dynamic(index_column='datetime', every='3h4m5s')
 .agg(pl.col(pl.Float64).mean())
)

In [ ]:
print(dd
 .sort('datetime')  
 .group_by_dynamic(index_column='datetime', every='1w', period='5d', 
                   start_by='monday')
 .agg(pl.col(pl.Float64).mean(),
     cfs_range=(pl.col('cfs').max() - pl.col('cfs').min()))
)

In [ ]:
print(dd
 .sort('datetime')  
 .group_by_dynamic(index_column='datetime', every='1w', period='2d', 
                   start_by='saturday')
 .agg(pl.col(pl.Float64).mean(),
     cfs_range=(pl.col('cfs').max() - pl.col('cfs').min()))
)

In [ ]:
print(dd
 .with_columns(year_mean_cfs=pl.col('cfs').mean().over(
     pl.col('datetime').dt.truncate('1y')
 ))
 .with_columns(pct_of_avg=(pl.col('cfs') / pl.col('year_mean_cfs'))
     .mul(100).round(2))
)

In [ ]:
print(dd
 .with_columns(year=pl.col('datetime').dt.year(),
               quarter=pl.col('datetime').dt.quarter())
 .with_columns(pl.col(['cfs', 'gage_height']).mean().over('year')
                .name.suffix('_mean_year'),
               pl.col(['cfs', 'gage_height']).mean().over('quarter')
                .name.suffix('_mean_quarter'))                   
)

In [ ]:
print(dd
 .filter((pl.col('datetime').dt.date() == datetime.date(2018, 9, 12))
         & pl.col('datetime').dt.hour().is_between(10, 11))
 .sort('datetime')
 .group_by_dynamic('datetime', every='30m', period='1h',
                   closed='left', include_boundaries=True)
 .agg(observed_at=pl.col('datetime'), mean_cfs=pl.col('cfs').mean())
)

In [ ]:
bucket_readings = (dd
 .filter((pl.col('datetime').dt.date() == datetime.date(2018, 9, 12))
         & pl.col('datetime').dt.hour().is_between(11, 14))
 .select('datetime', 'cfs')
 .sort('datetime')
)
print(bucket_readings
 .with_columns(hour_start=pl.col('datetime').dt.truncate('1h'),
               nearest_hour=pl.col('datetime').dt.round('1h'))
)

In [ ]:
fig

In [ ]:
print(bucket_readings
 .with_columns(hour_start=pl.col('datetime').dt.truncate('1h'))
 .group_by('hour_start')
 .agg(mean_cfs=pl.col('cfs').mean(), 
      readings=pl.col('cfs').count())
 .sort('hour_start')
)

In [ ]:
print(bucket_readings
 .group_by_dynamic('datetime', every='1h')
 .agg(mean_cfs=pl.col('cfs').mean(), 
      readings=pl.col('cfs').count())
)

In [ ]:
print(dd
 .with_columns(year=pl.col('datetime').dt.year(),
               month=pl.col('datetime').dt.month())
 .group_by('year', 'month')
 .agg(mean_cfs=pl.col('cfs').mean())
 .sort('year', 'month')
 .with_columns(report_date=pl.date('year', 'month', 1))
)

In [ ]:
expected_times = pl.DataFrame({'datetime': pl.datetime_range(
    datetime.datetime(2018, 9, 12), datetime.datetime(2018, 9, 13),
    interval='15m', closed='left', time_zone='America/Denver',
    eager=True)})
print(expected_times)

In [ ]:
_ = pl.Config.set_tbl_rows(8)

In [ ]:
print(expected_times
 .join(dd.with_columns(recorded=pl.lit(True)), on='datetime', how='left')
 .with_columns(cfs_updated=pl.col('cfs').fill_null(pl.col('cfs').mean()))
 .filter(pl.col('datetime').is_between(
     datetime.datetime(2018, 9, 12, 12, 15,
     tzinfo=ZoneInfo('America/Denver')),
     datetime.datetime(2018, 9, 12, 14, 0,
     tzinfo=ZoneInfo('America/Denver'))))
 .select('datetime', 'cfs', 'cfs_updated', 'recorded')
)

In [ ]:
_ = pl.Config.set_tbl_rows(6)

In [ ]:
monthly_report = (dd
 .filter((pl.col('datetime').dt.year() == 2018)
         & (pl.col('datetime').dt.month() >= 7))
 .sort('datetime')
 .group_by_dynamic('datetime', every='1mo')
 .agg(mean_cfs=pl.col('cfs').mean(),
      max_gage_height=pl.col('gage_height').max())
 .with_columns(month=pl.col('datetime').dt.date())
 .drop('datetime')
 .with_columns(next_month=pl.col('month').dt.offset_by('1mo'))
 .with_columns(publish_date=pl.col('next_month')
     .dt.add_business_days(0, roll='forward'))
 .with_columns(month_label=pl.col('month').dt.strftime('%m-%Y'))
)
print(monthly_report
 .select('month_label', 'mean_cfs', 'max_gage_height', 'publish_date')
)

In [ ]:
report_holidays = [datetime.date(2018, 9, 3), datetime.date(2019, 1, 1)]
print(monthly_report
 .with_columns(publish_with_holidays=pl.col('next_month').dt
     .add_business_days(
     0, roll='forward', holidays=report_holidays))
 .select('month_label', 'publish_date', 'publish_with_holidays')
)

In [ ]:
import holidays
us_2018 = holidays.country_holidays('US', years=2018, observed=True)

In [ ]:
_ = pl.Config.set_tbl_rows(12)

In [ ]:
print(pl.DataFrame(
    sorted(us_2018.items()), schema=['holiday_date', 'holiday'],
    orient='row')
)

In [ ]:
_ = pl.Config.set_tbl_rows(6)

In [ ]:
first_year = monthly_report['month'].min().year
last_year = monthly_report['next_month'].max().year
us_calendar = holidays.country_holidays(
    'US', years=range(first_year, last_year + 2), observed=True)

print(monthly_report
 .with_columns(publish_us_calendar=pl.col('next_month').dt
     .add_business_days(
     0, roll='forward', holidays=sorted(us_calendar)))
 .select('month_label', 'publish_date', 'publish_us_calendar')
)

In [ ]:
snake_raw = (pl.read_csv('data/nwis.waterservices.usgs.gov.txt', 
                     skip_rows=27, separator='\t',
                     skip_rows_after_header=1,
                     schema_overrides={'site_no': pl.String}))
snake = (tweak_river(snake_raw
           .with_columns(cfs=pl.lit(None).cast(pl.Float64)),
      cfs_col='cfs', gage_height_col='319803_00065') 
 )

In [ ]:
common_cols = ['agency_cd', 'site_no', 'datetime', 
               'cfs', 'gage_height', 'source']
print(snake
 .with_columns(source=pl.lit('snake'))
 .select(common_cols)
 .vstack(dd
         .with_columns(source=pl.lit('devil'))
         .select(common_cols)
         )
)

In [ ]:
common_cols = ['agency_cd', 'site_no', 'datetime', 
               'cfs', 'gage_height', 'source']
print(snake
 .with_columns(source=pl.lit('snake'))
 .select(common_cols)
 .vstack(dd
         .with_columns(source=pl.lit('devil'))
         .select(common_cols)
         )
 .sort('datetime')
 .group_by_dynamic(index_column='datetime', every='1mo', 
                   group_by='source')
 .agg(pl.col('gage_height').mean())
)

In [ ]:
common_cols = ['agency_cd', 'site_no', 'datetime', 
               'cfs', 'gage_height', 'source']
print(snake
 .with_columns(source=pl.lit('snake'))
 .select(common_cols)
 .vstack(dd
         .with_columns(source=pl.lit('devil'))
         .select(common_cols)
         )
 .group_by([pl.col('datetime').dt.truncate('1mo'), 'source'])
 .agg(pl.col('gage_height').mean())
 .sort('datetime', 'source')
)

In [ ]:
common_cols = ['agency_cd', 'site_no', 'datetime', 
               'cfs', 'gage_height', 'source']
print(snake
 .with_columns(source=pl.lit('snake'))
 .select(common_cols)
 .vstack(dd
         .with_columns(source=pl.lit('devil'))
         .select(common_cols)
         )
 .sort('datetime')
 .group_by_dynamic(index_column='datetime', every='1mo', 
                   group_by='source')
 .agg(pl.col('gage_height').mean())
 .pivot(on='source', index='datetime', values='gage_height')
)

In [ ]:
common_cols = ['agency_cd', 'site_no', 'datetime', 
               'cfs', 'gage_height', 'source']
plot_data = (snake
 .with_columns(source=pl.lit('snake'))
 .select(common_cols)
 .vstack(dd
         .with_columns(source=pl.lit('devil'))
         .select(common_cols)
         )
 .sort('datetime')
 .group_by_dynamic(index_column='datetime', every='1mo',
     group_by='source')
 .agg(pl.col('gage_height').mean())
 .pivot(on='source', index='datetime', values='gage_height')
 .filter(~pl.all_horizontal(pl.col('devil', 'snake').is_null()))
 )
from matplotlib import patheffects

with plt.rc_context(TS_STYLE):
    fig, ax = plt.subplots(figsize=(6, 4))
    for column, label, color, style in [
        ('devil', 'Dirty Devil River', TS_BLUE, '-'),
        ('snake', 'Snake River', '#b0b0b0', '-'),
    ]:
        ax.plot(plot_data['datetime'].to_list(),
        plot_data[column].to_numpy(),
                color=color, linestyle=style, linewidth=1.3)
        last = plot_data.filter(pl.col(column).is_not_null()).tail(1)
        ax.annotate(label, xy=(last['datetime'][0], last[column][0]),
                    xytext=(6, 0), textcoords='offset points',
                    color=color, fontsize=12, va='center',
                    path_effects=[patheffects.withStroke(linewidth=3,
                    foreground='white')])
    ax.set(ylabel='Monthly mean gage height (ft)',
           xlim=(plot_data['datetime'].min(),
                 datetime.datetime(2028, 6, 1,
                 tzinfo=ZoneInfo('America/Denver'))))
    ax.xaxis.set_major_locator(mdates.YearLocator(4,
    tz=ZoneInfo('America/Denver')))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y',
    tz=ZoneInfo('America/Denver')))
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color='#dddddd', linewidth=.5)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='both', length=0)
    fig.suptitle('Monthly gage height by river', y=.96)
    fig.text(.5, .88, ".group_by_dynamic(every='1mo', group_by='source')",
             ha='center', fontsize=9, fontfamily=TS_CODE_FONT)
    fig.text(.13, .04, 'Source: USGS stations 09333500 and 13022500',
    fontsize=8)
    fig.subplots_adjust(left=.13, right=.97, bottom=.16, top=.8)

In [ ]:
print(dd
 .filter((pl.col('datetime').dt.date() == datetime.date(2018, 9, 12))
         & pl.col('datetime').dt.hour().is_between(12, 14))
 .sort('datetime')
 .with_columns(previous_cfs=pl.col('cfs').shift(1),
               change_cfs=pl.col('cfs').diff(),
               elapsed_minutes=pl.col('datetime').diff().dt
               .total_minutes())
 .select('datetime', 'cfs', 'previous_cfs', 'change_cfs',
     'elapsed_minutes')
)

In [ ]:
gap_readings = (dd
 .filter((pl.col('datetime').dt.date() == datetime.date(2018, 9, 12))
         & pl.col('datetime').dt.hour().is_between(10, 15))
 .select('datetime', 'cfs')
 .sort('datetime')
)
print(gap_readings
 .with_columns(four_readings=pl.col('cfs').rolling_mean(4, min_samples=1),
               one_hour=pl.col('cfs').rolling_mean_by('datetime', '1h',
               min_samples=1))
 .filter(pl.col('datetime').dt.strftime('%H:%M').is_in(
     ['12:15', '12:30', '13:45', '14:00', '14:15', '14:30']))
)

In [ ]:
print(dd
 .sort('datetime')  
 .group_by_dynamic(index_column='datetime', every='1mo')
 .agg(pl.col('cfs').mean())
 .with_columns(mean_cfs_3mo=pl.col('cfs').mean()
               .rolling('datetime', period='3mo'))
)

In [ ]:
plot_data = (dd
 .sort('datetime')  
 .group_by_dynamic(index_column='datetime', every='1mo')
 .agg(pl.col('cfs').mean())
 .with_columns(mean_cfs_3mo=pl.col('cfs').mean()
               .rolling('datetime', period='3mo'))
 )
fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
for column, label, style in [
    ('cfs', 'Monthly mean', '-'),
    ('mean_cfs_3mo', 'Three-month mean', '--'),
]:
    _ = ax.plot(plot_data['datetime'], plot_data[column],
                label=label, linestyle=style)
_ = ax.set(xlabel='Date', ylabel='Discharge (cfs)')
_ = ax.legend(fontsize=7)
ax.set_xlim(plot_data['datetime'].min(), 
   plot_data.select(pl.col('datetime').min().dt.offset_by('7y')).item())
ax.set_ylim(0, 1_000)
fig.autofmt_xdate()
fig

In [ ]:
print(snake
 .select('datetime', 'gage_height', source=pl.lit('Snake River'))
 .vstack(dd
     .select('datetime', 'gage_height',
     source=pl.lit('Dirty Devil River')))
 .sort('datetime')
 .group_by_dynamic('datetime', every='1mo', group_by='source')
 .agg(mean_height=pl.col('gage_height').mean())
 .sort('source', 'datetime')
 .with_columns(mean_height_3mo=pl.col('mean_height').mean()
     .rolling('datetime', period='3mo').over('source'))
 .filter(pl.col('datetime').dt.year() == 2018)
)

In [ ]:
daily_flow = (dd
 .sort('datetime')
 .group_by_dynamic('datetime', every='1d')
 .agg(mean_cfs=pl.col('cfs').mean())
)
print(daily_flow
 .with_columns(mad_cfs=pl.col('mean_cfs').rolling_map(
     lambda window: (window - window.median()).abs().median(),
     window_size=7, min_samples=7),
     std_cfs=pl.col('mean_cfs').rolling_std(
         window_size=7, min_samples=7, ddof=0))
 .filter((pl.col('datetime').dt.year() == 2018)
         & (pl.col('datetime').dt.month() == 7))
 .select('datetime', 'mean_cfs', 'mad_cfs', 'std_cfs')
)

In [ ]:
july_seventh_readings = (dd
 .filter((pl.col('datetime').dt.date() == datetime.date(2018, 7, 7))
         & pl.col('datetime').dt.hour().is_between(12, 18))
 .select('datetime', 'cfs')
 .sort('datetime')
)
print(july_seventh_readings)

In [ ]:
print(july_seventh_readings
 .with_columns(known=pl.col('cfs').is_not_null().cast(pl.UInt32)
     .rolling_sum(4, min_samples=1),
               mean_one=pl.col('cfs').rolling_mean(4, min_samples=1),
               mean_four=pl.col('cfs').rolling_mean(4, min_samples=4))
 .filter(pl.col('datetime').dt.strftime('%H:%M').is_in(
     ['13:00', '13:15', '14:00', '17:30', '17:45', '18:30']))
)

In [ ]:
print(dd
 .sort('datetime')
 .group_by_dynamic('datetime', every='1d')
 .agg(max_cfs=pl.col('cfs').max())
 .upsample('datetime', every='1d')
 .with_columns(peak=pl.col('max_cfs').peak_max(),
               previous=pl.col('max_cfs').shift(1),
               following=pl.col('max_cfs').shift(-1))
 .filter(pl.col('peak') & pl.col('previous').is_not_null()
     & pl.col('following').is_not_null())
 .filter(pl.col('datetime').dt.year() == 2018)
 .select('datetime', 'previous', 'max_cfs', 'following')
 # only show September 2018
 .filter(pl.col('datetime').dt.month() == 9)
)

In [ ]:
print(dd
.filter(pl.col('cfs').is_null())
)

In [ ]:
denver = ZoneInfo('America/Denver')
jul_7 = datetime.datetime(2018, 7, 7).replace(tzinfo=denver)
jul_9 = datetime.datetime(2018, 7, 9).replace(tzinfo=denver)

print(dd
 .filter(pl.col('datetime').is_between(jul_7, jul_9))
 .filter(pl.col('cfs').is_null())
)

In [ ]:
print(dd
 .filter(pl.col('datetime').is_between(jul_7, jul_9))
 .sort('datetime')
 .with_columns(
     fill0=pl.col('cfs').fill_null(0),
     forward=pl.col('cfs').fill_null(strategy='forward'),
     backward=pl.col('cfs').fill_null(strategy='backward'),
     interpolated=pl.col('cfs').interpolate())
 .filter(pl.col('datetime').dt.date() == jul_7.date())
 .filter(pl.col('datetime').dt.strftime('%H:%M').is_in(
     ['13:00', '13:15', '14:00', '17:30', '17:45']))
 .select(time=pl.col('datetime').dt.strftime('%H:%M'),
         cfs=pl.col('cfs'), fill0=pl.col('fill0'),
         forward=pl.col('forward'), backward=pl.col('backward'),
         interpolated=pl.col('interpolated'))
)

In [ ]:
interpolation_readings = (dd
 .filter(pl.col('datetime').dt.date().is_between(
     datetime.date(2007, 12, 14), datetime.date(2007, 12, 18)))
 .select('datetime', 'cfs')
 .sort('datetime')
)
print(interpolation_readings
 .with_columns(by_row=pl.col('cfs').interpolate(),
               by_time=pl.col('cfs').interpolate_by('datetime'))
 .filter(pl.col('cfs').is_null())
)

In [ ]:
print(dd
 .filter((pl.col('datetime').dt.date() == datetime.date(2018, 9, 12))
         & (pl.col('datetime').dt.hour() == 10))
 .select('datetime', 'cfs')
 .sort('datetime')
 .upsample('datetime', every='5m')
 .interpolate()
 )

In [ ]:
offset_grid = pl.DataFrame({'datetime': pl.datetime_range(
    datetime.datetime(2018, 9, 12, 10, 2),
    datetime.datetime(2018, 9, 12, 11),
    interval='5m', closed='left', time_zone='America/Denver', eager=True,
)})
print(offset_grid
 .with_columns(requested=pl.lit(True))
 .join(dd
     .filter(pl.col('datetime').is_between(
         datetime.datetime(2018, 9, 12, 10,
         tzinfo=ZoneInfo('America/Denver')),
         datetime.datetime(2018, 9, 12, 11,
         tzinfo=ZoneInfo('America/Denver'))))
     .select('datetime', 'cfs'),
     on='datetime', how='full', coalesce=True)
 .sort('datetime')
 .with_columns(estimated_cfs=pl.col('cfs').interpolate_by('datetime'))
 .filter(pl.col('requested').fill_null(False))
 .select('datetime', 'estimated_cfs')
)

In [ ]:
print(dd
 .sort('datetime')
 .group_by_dynamic(index_column='datetime', every='30m')
 .agg(pl.col('cfs').mean())
)

In [ ]:
station_grid = (snake
 .select('datetime', 'gage_height', source=pl.lit('Snake River'))
 .vstack(dd
     .select('datetime', 'gage_height',
     source=pl.lit('Dirty Devil River')))
 .filter(pl.col('datetime').is_between(
     datetime.datetime(2018, 9, 12, 10,
     tzinfo=ZoneInfo('America/Denver')),
     datetime.datetime(2018, 9, 12, 12,
     tzinfo=ZoneInfo('America/Denver'))))
 .sort('source', 'datetime')
 .upsample('datetime', every='5m', group_by='source', maintain_order=True)
 .with_columns(height_filled=pl.col('gage_height').interpolate()
     .over('source'))
)

In [ ]:
_ = pl.Config.set_tbl_rows(10)

In [ ]:
print(station_grid
 .filter(pl.col('datetime').dt.hour().eq(10)
         & pl.col('datetime').dt.minute().le(20))
)

In [ ]:
_ = pl.Config.set_tbl_rows(6)

In [ ]:
url = 'data/hanksville.csv'

raw_temp = pl.read_csv(url)
print(raw_temp
        .select(pl.col('DATE').str.to_datetime()
                  .dt.replace_time_zone('America/Denver'),
                'PRCP', 'TMIN', 'TMAX', 'TOBS')
    )

In [ ]:
def tweak_temp(df_: pl.DataFrame) -> pl.DataFrame:
    return (df_  # Capture
        .select('DATE', 'PRCP', 'TMIN', 'TMAX', 'TOBS')  # Limit
        # Edit: retain the source column names
        .with_columns(  # Audit: parse and localize the daily timestamp
            pl.col('DATE').str.to_datetime()
                .dt.replace_time_zone('America/Denver')
        )
        # Null: retain missing measurements
        # Engineer: no additional features needed
        .select('DATE', 'PRCP', 'TMIN', 'TMAX', 'TOBS')  # Restrict
    )

print(tweak_temp(raw_temp))

In [ ]:
dd_daily = (dd
 .sort('datetime')
 .group_by_dynamic(index_column='datetime', every='1d')
 .agg(pl.col('gage_height', 'cfs').mean())
)

print(dd_daily)

In [ ]:
both = (tweak_temp(raw_temp)
 .join(dd_daily, left_on='DATE', right_on='datetime',
       validate='1:1')
 )

print(both)

In [ ]:
weather_days = (tweak_temp(raw_temp)
 .filter(pl.col('DATE').dt.date().is_between(
     datetime.date(2018, 8, 12), datetime.date(2018, 8, 16)))
 .select('DATE', 'TMAX', 'PRCP')
 .with_columns(cutoff=pl.col('DATE').dt.offset_by('1d'))
 .sort('cutoff')
)
print(weather_days)

In [ ]:
observations = (dd
 .filter(pl.col('cfs').is_not_null())
 .select(observed_at=pl.col('datetime'), cfs=pl.col('cfs'))
 .sort('observed_at')
)
print(observations)

In [ ]:
daily_report = (weather_days
 .join_asof(observations, left_on='cutoff', right_on='observed_at',
     strategy='backward', tolerance='20m', allow_exact_matches=False,
     coalesce=False)
 .with_columns(age_minutes=(pl.col('cutoff') - pl.col('observed_at'))
     .dt.total_minutes())
)
print(daily_report
 .select(day=pl.col('DATE').dt.date(),
         max_temp_f=pl.col('TMAX'), precipitation_in=pl.col('PRCP'),
         end_of_day_cfs=pl.col('cfs'), age_minutes=pl.col('age_minutes'))
)

In [ ]:
print(daily_report
 .filter(pl.col('observed_at').is_not_null())
)

In [ ]:
year_agg = (both
    .with_columns(day_of_year=pl.col('DATE').dt.ordinal_day(),
                  year=pl.col('DATE').dt.year())
    .pivot(index='day_of_year', on='year', values='TOBS')
    .sort('day_of_year'))
latest = str(both['DATE'].dt.year().max())
with plt.rc_context(TS_STYLE):
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    for year in year_agg.columns[1:]:
        highlighted = year == latest
        ax.plot(year_agg['day_of_year'],
            year_agg[year].rolling_median(7),
            color=TS_BLUE if highlighted else '#cccccc',
            linewidth=1.8 if highlighted else .6,
            zorder=3 if highlighted else 1,
            label=latest if highlighted else None)
    ax.set(xlabel='Day of year', ylabel='Temperature (°F)',
        xlim=(1, 366), xticks=[1, 60, 120, 180, 240, 300, 366])
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.spines['bottom'].set_color('#bbbbbb')
    ax.tick_params(length=0)
    ax.set_axisbelow(True)
    ax.grid(axis='y', color='#eeeeee', linewidth=.5)
    ax.legend(frameon=False, loc='upper left')
    fig.text(.13, .95, f'{latest} temperature and earlier years',
        fontsize=12, ha='left', va='top')
    fig.text(.13, .88, 'Earlier years in gray · 7-observation medians',
        fontsize=9, color='#555555')
    fig.text(.13, .03, 'Source: NOAA Hanksville weather; matched dates',
        fontsize=8, color='#555555')
    fig.subplots_adjust(left=.13, right=.97, bottom=.17, top=.80)
fig

In [ ]:
def plot_year_last_year_median(
    df: pl.DataFrame, col: str, upper_limit: float | None = None
) -> plt.Figure:
    year_agg = (df
        .with_columns(day_of_year=pl.col('DATE').dt.ordinal_day(),
                      year=pl.col('DATE').dt.year())
        .pivot(index='day_of_year', on='year', values=col)
        .sort('day_of_year'))
    latest = str(df['DATE'].dt.year().max())
    median = (df
        .with_columns(day_of_year=pl.col('DATE').dt.ordinal_day())
        .group_by('day_of_year')
        .agg(pl.col(col).median())
        .sort('day_of_year'))
    with plt.rc_context(TS_STYLE):
        fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
        for year in year_agg.columns[1:]:
            highlighted = year == latest
            ax.plot(year_agg['day_of_year'],
                year_agg[year].rolling_median(7),
                color=TS_BLUE if highlighted else '#cccccc',
                linewidth=1.8 if highlighted else .6,
                zorder=4 if highlighted else 1,
                label=latest if highlighted else None)
        ax.plot(median['day_of_year'], median[col].rolling_median(7),
            color='#444444', linestyle='--', linewidth=1.2,
            label='Across-year median', zorder=3)
        ylabel = 'Discharge (ft³/s)' if col == 'cfs' else col
        ax.set(xlabel='Day of year', ylabel=ylabel,
            ylim=(0, upper_limit), xlim=(1, 366),
            xticks=[1, 60, 120, 180, 240, 300, 366])
        ax.spines[['top', 'right', 'left']].set_visible(False)
        ax.spines['bottom'].set_color('#bbbbbb')
        ax.tick_params(length=0)
        ax.set_axisbelow(True)
        ax.grid(axis='y', color='#eeeeee', linewidth=.5)
        ax.legend(frameon=False, loc='upper right', fontsize=9)
        fig.text(.13, .95,
            f'{latest} River Discharge and Earlier Years',
            fontsize=12, ha='left', va='top')
        fig.text(.13, .88,
            'Earlier years in gray · 7-observation medians',
            fontsize=9, color='#555555')
        fig.text(.13, .03,
            'Source: USGS Dirty Devil River; weather-matched dates',
            fontsize=8, color='#555555')
        fig.subplots_adjust(left=.13, right=.97, bottom=.17, top=.80)
    return fig

In [ ]:
fig = plot_year_last_year_median(both, 'cfs', upper_limit=400)
fig

In [ ]:
decay = (dd
 .filter((pl.col('datetime').dt.date() == datetime.date(2018, 9, 12))
         & pl.col('datetime').dt.hour().is_between(10, 15))
 .select('datetime', 'cfs')
 .sort('datetime')
)
print(decay
 .with_columns(by_observation=pl.col('cfs').ewm_mean(alpha=0.5,
     adjust=False),
               by_elapsed_time=pl.col('cfs').ewm_mean_by('datetime',
               half_life='15m'))
 .filter(pl.col('datetime').dt.strftime('%H:%M').is_in(
     ['12:15', '12:30', '13:45', '14:00', '14:15', '14:30']))
)

In [ ]:
print(july_seventh_readings
 .with_columns(count_gap=pl.col('cfs').ewm_mean(alpha=0.5,
     ignore_nulls=False),
               ignore_gap=pl.col('cfs').ewm_mean(alpha=0.5,
               ignore_nulls=True))
)

# Data Import and Export

In [ ]:
import io
import json
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import polars as pl
import xgboost as xgb
from highlight_text import fig_text
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text

In [ ]:
_ = pl.Config.set_tbl_width_chars(73)
_ = pl.Config.set_tbl_rows(6)
_ = pl.Config.set_tbl_cols(7)
_ = pl.Config.set_float_precision(2)

In [ ]:
output = Path('build/examples/io-common')
output.mkdir(parents=True, exist_ok=True)

In [ ]:
autos = pl.read_csv(
    'data/vehicles.csv',
    use_pyarrow=True,
    columns=['id', 'year', 'make', 'model', 'displ', 'cylinders',
             'fuelType', 'city08', 'highway08'],
)
print(autos)

In [ ]:
autos.schema

In [ ]:
print(autos.cast({
    'id': pl.Int32,
    'year': pl.Int16,
    'cylinders': pl.Int16,
    'city08': pl.Int16,
    'highway08': pl.Int16,
})
.select('id', 'year', 'cylinders', 'city08', 'highway08'))

In [ ]:
missing_csv = (
    autos
    .filter(pl.col('id').is_in([16423, 16424, 17328]))
    .select('id', 'model', 'cylinders')
    .write_csv(separator=';', null_value='NA')
)
print(missing_csv)

In [ ]:
pl.read_csv(io.StringIO(missing_csv), separator=';').schema

In [ ]:
print(pl.read_csv(
    io.StringIO(missing_csv),
    separator=';',
    null_values='NA',
    schema_overrides={'id': pl.Int32, 'cylinders': pl.Int16},
))

In [ ]:
export_columns = ['id', 'year', 'make', 'model', 'city08', 'highway08']
(autos
 .select(export_columns)
 .write_csv(output / 'vehicles.csv')
)
print(pl.read_csv(output / 'vehicles.csv'))

In [ ]:
print(pl.scan_csv(
    output / 'vehicles.csv',
    schema_overrides=autos.select(export_columns).schema,
 )
 .filter((pl.col('year') == 2020) & (pl.col('make') == 'Ford'))
 .select('id', 'model', 'city08', 'highway08')
 .explain(optimized=True)
)

In [ ]:
print(pl.scan_csv(
    output / 'vehicles.csv',
    schema_overrides=autos.select(export_columns).schema,
 )
 .filter((pl.col('year') == 2020) & (pl.col('make') == 'Ford'))
 .select('id', 'model', 'city08', 'highway08')
 .collect()
)

In [ ]:
print(autos
 .filter(pl.col('year') == 2020)
 .group_by('make')
 .agg(city_mpg=pl.col('city08').mean())
 .sort('make')
 .write_csv(float_precision=1)
)

In [ ]:
autos.write_parquet(output / 'vehicles.parquet')
pl.read_parquet(output / 'vehicles.parquet').schema

In [ ]:
print(pl.scan_parquet(output / 'vehicles.parquet')
 .filter(
     (pl.col('year') >= 2018)
     & (pl.col('fuelType') == 'Regular')
     & (pl.col('city08') >= 30)
 )
 .select('id', 'year', 'make', 'model', 'city08')
 .collect()
)

In [ ]:
(pl.scan_parquet(output / 'vehicles.parquet')
 .filter(
     (pl.col('year') >= 2018)
     & (pl.col('fuelType') == 'Regular')
     & (pl.col('city08') >= 30)
 )
 .select('id', 'year', 'make', 'model', 'city08')
 .sink_parquet(output / 'efficient-vehicles.parquet')
)
print(pl.read_parquet(output / 'efficient-vehicles.parquet'))

In [ ]:
yearly_directory = output / 'yearly'
yearly_directory.mkdir(exist_ok=True)
for year in (2018, 2019, 2020):
    (autos
     .filter(pl.col('year') == year)
     .select(export_columns)
     .write_parquet(yearly_directory / f'vehicles-{year}.parquet')
    )

In [ ]:
print(pl.scan_parquet(yearly_directory / 'vehicles-*.parquet')
 .group_by('year')
 .agg(records=pl.len())
 #.sort('year')
 .collect()
)

In [ ]:
(autos
 .filter(pl.col('year') == 2020)
 .select(*export_columns, 'displ')
 .write_parquet(yearly_directory / 'vehicles-2020.parquet')
)
print(pl.scan_parquet(
    yearly_directory / 'vehicles-*.parquet',
    schema=autos.select(*export_columns, 'displ').schema,
    missing_columns='insert',
 )
 .group_by('year')
 .agg(records=pl.len(), known_displacement=pl.col('displ').count())
 .sort('year')
 .collect()
)

In [ ]:
hive_directory = output / 'vehicles-hive'
(autos
 .filter(pl.col('year').is_between(2018, 2020))
 .select(export_columns)
 .write_parquet(hive_directory, partition_by='year')
)

In [ ]:
print(pl.scan_parquet(hive_directory)
 .group_by('year')
 .agg(records=pl.len())
 .sort('year')
 .collect()
)

In [ ]:
print(pl.scan_parquet(
    hive_directory / 'year=*/*.parquet',
    hive_partitioning=True,
 )
 .filter(pl.col('year') == 2020)
 .group_by('year')
 .agg(records=pl.len())
 .collect()
)

In [ ]:
sample_ids = [41468, 41469, 41470]
json_vehicles = (autos
 .filter(pl.col('id').is_in(sample_ids))
 .select(export_columns)
 .sort('id')
)
json_vehicles.write_json(output / 'vehicles.json')
print(
    (output / 'vehicles.json').read_text()
)

In [ ]:
print(pl.read_json(output / 'vehicles.json', schema=json_vehicles.schema))

In [ ]:
json_vehicles.write_ndjson(output / 'vehicles.ndjson')
print(
    (output / 'vehicles.ndjson').read_text()
)

In [ ]:
print(pl.scan_ndjson(output / 'vehicles.ndjson')
 .cast(json_vehicles.schema)
 .select('id', 'model', 'city08', 'highway08')
 .collect()
)

In [ ]:
vehicle_payload = {
    'data': {
        'vehicles': [
            {
                'id': row['id'],
                'vehicle': {
                    'year': row['year'],
                    'identity': {
                        'make': row['make'],
                        'model': row['model'],
                    },
                    'engine': {
                        'displacement_l': row['displ'],
                        'cylinders': row['cylinders'],
                    },
                },
                'economy': {
                    'measurements': [
                        {'cycle': 'city', 'mpg': row['city08']},
                        {'cycle': 'highway', 'mpg': row['highway08']},
                    ],
                },
            }
            for row in (
                autos
                .filter(pl.col('id').is_in(sample_ids))
                .sort('id')
                .to_dicts()
            )
        ],
    },
}
_ = (output / 'nested-vehicles.json').write_text(
    json.dumps(vehicle_payload, indent=2), encoding='utf-8',
)
print(json.dumps(vehicle_payload['data']['vehicles'][0], indent=2))

In [ ]:
payload = json.loads(
    (output / 'nested-vehicles.json').read_text(encoding='utf-8')
)
print(pl.json_normalize(payload['data']['vehicles'], separator='_')
 .select(
     'id',
     model=pl.col('vehicle_identity_model'),
     displ=pl.col('vehicle_engine_displacement_l'),
 )
)

In [ ]:
print(pl.json_normalize(payload['data']['vehicles'], separator='_')
 .select('id', 'economy_measurements')
)

In [ ]:
print(pl.json_normalize(payload['data']['vehicles'], separator='_')
 .select('id', model=pl.col('vehicle_identity_model'),
         measurements=pl.col('economy_measurements'))
 .explode('measurements')
 .unnest('measurements')
 .sort('id', 'cycle')
)

In [ ]:
print(pl.read_json(output / 'nested-vehicles.json')
 .select(pl.col('data').struct.field('vehicles'))
 .explode('vehicles')
 .unnest('vehicles')
)

In [ ]:
print(pl.read_json(output / 'nested-vehicles.json')
 .select(pl.col('data').struct.field('vehicles'))
 .explode('vehicles')
 .unnest('vehicles')
 .select(
     'id',
     pl.col('vehicle').struct.field('identity').struct.field('model'),
     pl.col('economy').struct.field('measurements'),
 )
 .explode('measurements')
 .unnest('measurements')
 .sort('id', 'cycle')
)

In [ ]:
_ = (autos
 .filter((pl.col('year') == 2020) & (pl.col('make') == 'Ford'))
 .select(export_columns)
 .write_excel(
     output / 'ford-2020.xlsx', worksheet='Vehicles', autofit=True,
 )
)

In [ ]:
print(pl.read_excel(
    output / 'ford-2020.xlsx',
    sheet_name='Vehicles',
    columns=['id', 'model', 'city08', 'highway08'],
    schema_overrides={
        'id': pl.Int32, 'city08': pl.Int16, 'highway08': pl.Int16,
    },
))

In [ ]:
_ = (autos
 .filter(pl.col('year') == 2020)
 .group_by('make')
 .agg(records=pl.len(), city_mpg=pl.col('city08').mean())
 .sort('make')
 .write_excel(
     output / 'mileage-report.xlsx',
     worksheet='Mileage',
     column_formats={'city_mpg': '0.0'},
     autofit=True,
 )
)

In [ ]:
f150_mileage = (autos
 .filter((pl.col('make') == 'Ford') & pl.col('model').str
     .contains('F150'))
 .group_by('year')
 .agg(city_mpg=pl.col('city08').mean(),
      highway_mpg=pl.col('highway08').mean())
 .sort('year')
)
print(f150_mileage)

In [ ]:
import xlsxwriter

with xlsxwriter.Workbook(output / 'f150-mileage.xlsx') as workbook:
    worksheet = workbook.add_worksheet('F150')
    _ = f150_mileage.write_excel(
        workbook=workbook,
        worksheet=worksheet,
        column_formats={'city_mpg': '0.0', 'highway_mpg': '0.0'},
        autofit=True,
    )
    chart = workbook.add_chart({'type': 'line'})
    for column, label, color in (
        (1, 'City', '#0072B2'),
        (2, 'Highway', '#666666'),
    ):
        chart.add_series({
            'name': label,
            'categories': ['F150', 1, 0, f150_mileage.height, 0],
            'values': ['F150', 1, column, f150_mileage.height, column],
            'line': {'color': color, 'width': 2},
        })
    chart.set_title({'name': 'Ford F-150 fuel economy by model year'})
    chart.set_x_axis({'name': 'Model year', 'interval_unit': 5})
    chart.set_y_axis({'name': 'Mean fuel economy (MPG)', 'min': 0})
    chart.set_legend({'position': 'bottom'})
    chart.set_size({'width': 720, 'height': 420})
    worksheet.insert_chart('E2', chart)
    worksheet.write('E24',
    'Source: data/vehicles.csv; mean across F150 records per year.')

In [ ]:
engine = create_engine(f"sqlite:///{(output / 'vehicles.db').resolve()}")
_ = (autos
 .select(export_columns)
 .write_database('vehicles', connection=engine, if_table_exists='replace')
)

In [ ]:
with engine.connect() as connection:
    database_vehicles = pl.read_database(
        text('''
            SELECT id, year, make, model, city08, highway08
            FROM vehicles
            WHERE year = :year AND make = :make
            ORDER BY id
        '''),
        connection=connection,
        execute_options={'parameters': {'year': 2020, 'make': 'Ford'}},
        schema_overrides=autos.select(export_columns).schema,
    )
print(database_vehicles)

In [ ]:
with engine.connect() as connection:
    batch_sizes = [
        batch.height
        for batch in pl.read_database(
            text('''
                SELECT id, city08 FROM vehicles
                WHERE make = :make ORDER BY id
            '''),
            connection=connection,
            execute_options={'parameters': {'make': 'Ford'}},
            iter_batches=True,
            batch_size=1000,
        )
    ]
engine.dispose()
print(batch_sizes)

In [ ]:
vehicle_records = (autos
 .filter(pl.col('id').is_in(sample_ids))
 .select(export_columns)
 .to_dicts()
)
print(vehicle_records[0])

In [ ]:
print(pl.from_dicts(vehicle_records,
    schema=autos.select(export_columns).schema))

In [ ]:
vehicle_rows = (autos
 .filter(pl.col('id').is_in(sample_ids))
 .select(export_columns)
 .rows()
)
print(vehicle_rows[0])

In [ ]:
print(pl.DataFrame(
    vehicle_rows,
    schema=autos.select(export_columns).schema,
    orient='row',
))

In [ ]:
vehicles_for_pandas = autos.select('id', 'cylinders').rechunk()
pandas_vehicles = vehicles_for_pandas.to_pandas(
    use_pyarrow_extension_array=True,
)
print(pandas_vehicles.dtypes)

In [ ]:
polars_vehicles = pl.from_pandas(pandas_vehicles, rechunk=False)
print(polars_vehicles.schema)

In [ ]:
import pyarrow as pa

original_arrow = vehicles_for_pandas.to_arrow()
pandas_arrow = pa.Table.from_pandas(pandas_vehicles, preserve_index=False)
returned_arrow = polars_vehicles.to_arrow()

def buffer_layout(column):
    assert column.num_chunks == 1
    array = column.chunk(0)
    return (
        array.type, len(array), array.offset,
        tuple(None if buf is None else (buf.address, buf.size)
              for buf in array.buffers()),
    )

for name in vehicles_for_pandas.columns:
    original = buffer_layout(original_arrow[name])
    in_pandas = buffer_layout(pandas_arrow[name])
    returned = buffer_layout(returned_arrow[name])
    print(f'{name}: to pandas={original == in_pandas}, '
          f'back to Polars={original == returned}')
    assert original == in_pandas == returned

In [ ]:
print(pl.from_pandas(pandas_vehicles.set_index('id'), include_index=True))

In [ ]:
print(autos
 .filter(pl.col('id').is_in([16423, 16424, 17328]))
 .get_column('cylinders')
 .to_numpy()
)

In [ ]:
print(autos
 .filter(pl.col('id').is_in(sample_ids))
 .select('city08', 'highway08')
 .to_numpy()
)

In [ ]:
print(autos
 .get_column('cylinders')
 .drop_nulls()
 .rechunk()
 .to_numpy(allow_copy=False)
 .flags.writeable
)

In [ ]:
arrow_vehicles = (autos
 .filter(pl.col('id').is_in(sample_ids))
 .select(export_columns)
 .to_arrow()
)
print(arrow_vehicles.schema)

In [ ]:
with duckdb.connect() as connection:
    _ = connection.register('vehicle_export', arrow_vehicles)
    arrow_mileage = connection.sql('''
        SELECT id, model, highway08 - city08 AS mpg_gap
        FROM vehicle_export
        ORDER BY id
    ''').to_arrow_table()
print(arrow_mileage.schema)

In [ ]:
print(pl.from_arrow(arrow_mileage))

In [ ]:
(autos
 .select(export_columns)
 .write_ipc(output / 'vehicles.arrow')
)
print(pl.read_ipc(output / 'vehicles.arrow'))

In [ ]:
with duckdb.connect() as connection:
    _ = connection.register('vehicles', autos)
    yearly_mileage = connection.sql('''
        SELECT year, avg(city08) AS city_mpg
        FROM vehicles
        WHERE fuelType = 'Regular'
        GROUP BY year
        ORDER BY year
    ''').pl()
print(yearly_mileage)

In [ ]:
model_data = (autos
 .filter(pl.col('fuelType') == 'Regular')
 .select('id', 'make', 'model', 'year', 'displ', 'cylinders', 'city08')
 .drop_nulls()
)
train, test = train_test_split(model_data, test_size=.25, random_state=42)
feature_columns = ['year', 'displ', 'cylinders']
print(train.select(feature_columns))

In [ ]:
scaler = StandardScaler().set_output(transform='polars')
print(scaler.fit_transform(train.select(feature_columns)))

In [ ]:
print(scaler.transform(test.select(feature_columns)))

In [ ]:
pca_pipeline = make_pipeline(StandardScaler(), PCA(n_components=2))
_ = (pca_pipeline
 .set_output(transform='polars')
 .fit(train.select(feature_columns))
)
print(pca_pipeline
 .transform(test.select(feature_columns))
 .with_columns(test.get_column('id'))
 .select('id', 'pca0', 'pca1')
)

In [ ]:
make_dtype = pl.Enum(autos.get_column('make').unique().sort())
xgb_features = [*feature_columns, 'make']
xgb_train = train.select(xgb_features).with_columns(pl.col('make')
    .cast(make_dtype))
xgb_test = test.select(xgb_features).with_columns(pl.col('make')
    .cast(make_dtype))
print(xgb_train.schema)

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=60, max_depth=4, n_jobs=2, random_state=42,
    tree_method='hist', enable_categorical=True,
)
_ = model.fit(xgb_train, train.get_column('city08'))
print(model.score(xgb_test, test.get_column('city08')))

In [ ]:
predictions = (test
 .with_columns(pl.Series(
     'predicted_city', model.predict(xgb_test),
 ))
 .with_columns(
     residual=pl.col('city08') - pl.col('predicted_city'),
 )
)
print(predictions.select('id', 'model', 'city08', 'predicted_city',
    'residual'))

In [ ]:
new_vehicle = (pl.DataFrame({
    'model': ['F-150'],
    'year': [1982],
    'displ': [5.0],
    'cylinders': [8],
    'make': ['Ford'],
})
 .with_columns(pl.col('make').cast(make_dtype))
)
print(new_vehicle.with_columns(pl.Series(
    'predicted_city', model.predict(new_vehicle.select(xgb_features)),
)))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
_ = ax.scatter(
    predictions['predicted_city'], predictions['residual'],
    alpha=.08, s=7, color='#245d83', rasterized=True,
)
_ = ax.axhline(0, color='#666666', linewidth=.8)
_ = ax.set_xlabel('Predicted city MPG')
_ = ax.set_ylabel('Actual minus predicted MPG')
ax.yaxis.set_label_position('right')
ax.yaxis.tick_right()
ax.spines[['top', 'left']].set_visible(False)
_ = fig_text(
    .12, .98, 'Compare mileage predictions with the test records',
    fontsize=11, va='top',
)
_ = fig_text(
    .12, .03,
    'Source: EPA vehicle data; regular gasoline; held-out test rows',
    fontsize=7,
)
fig.subplots_adjust(top=.85, bottom=.2, right=.88)
plt.close(fig)

# Extending Polars with Python and Rust

In [ ]:
import math
import re

import numpy as np
import polars as pl
from polars.testing import assert_frame_equal

In [ ]:
_ = pl.Config.set_tbl_rows(10)
_ = pl.Config.set_tbl_cols(8)
_ = pl.Config.set_tbl_width_chars(80)
_ = pl.Config.set_float_precision(4)

In [ ]:
vehicles = pl.read_csv(
    'data/vehicles.csv', use_pyarrow=True,
    columns=['id', 'make', 'model', 'year', 'displ', 'cylinders',
             'eng_dscr', 'trany'],
)
vehicle_ids = [10275, 10276, 10277, 10278, 41468, 41469, 41470]
print(vehicles
  .filter(pl.col('id').is_in(vehicle_ids))
  .sort('id')
  .select('id', 'make', 'model', 'trany')
)

In [ ]:
print(vehicles
  .select('trany')
  .unique()
  .sort('trany')
)

In [ ]:
GEAR_PATTERN = (r'^(?:Manual|Automatic) (?:'  # Type; either format.
    r'([0-9]+)-spd(?: Doubled)?|'  # Capture 1: written speed count.
    r'\((?:AM-S|AV-S|AM|A|L|S)([0-9]+)\))$')  # Capture 2: code count.


def parse_gears(label):
    if label is None:
        return None
    match = re.fullmatch(GEAR_PATTERN, label)
    if match is None:
        return None
    count = int(match.group(1) or match.group(2))
    return count if 1 <= count <= 10 else None


print(vehicles
  .select('trany')
  .unique()
  .sort('trany')
  .with_columns(
    reported_gears=pl.col('trany').map_elements(
        parse_gears, return_dtype=pl.UInt32,
    ),
  )
)

In [ ]:
def gears_expr(column):
    expr = pl.col(column) if isinstance(column, str) else column
    count = pl.coalesce(
        expr.str.extract(GEAR_PATTERN, 1), expr.str.extract(GEAR_PATTERN,
        2),
    ).cast(pl.UInt32, strict=False)
    return pl.when(count.is_between(1, 10)).then(count).otherwise(None)


print(vehicles
  .select('trany')
  .unique()
  .sort('trany')
  .with_columns(
    reported_gears=gears_expr('trany'),
  )
)

In [ ]:
assert_frame_equal(
    vehicles
      .select(reported_gears=gears_expr('trany')),
    vehicles
      .select(reported_gears=pl.col('trany').map_elements(
        parse_gears, return_dtype=pl.UInt32,
      )),
)

In [ ]:
%%timeit 
(vehicles
  .select(pl.col('trany').map_elements(parse_gears,
      return_dtype=pl.UInt32))
)

In [ ]:
%%timeit 
(vehicles
  .select(gears_expr('trany'))
)

In [ ]:
@pl.api.register_expr_namespace('vehicle')
class VehicleNamespace:
    def __init__(self, expr):
        self._expr = expr

    def reported_gears(self):
        return gears_expr(self._expr)

In [ ]:
print(vehicles
  .filter(pl.col('id').is_in(vehicle_ids))
  .select(
    'id', 'trany',
    reported_gears=pl.col('trany').vehicle.reported_gears(),
  )
)

In [ ]:
@pl.api.register_dataframe_namespace('vehicle')
class VehicleFrameNamespace:
    def __init__(self, frame):
        self._frame = frame

    def with_reported_gears(self):
        return (self._frame
          .with_columns(
            reported_gears=pl.col('trany').vehicle.reported_gears(),
          )
        )

In [ ]:
print(vehicles
  .filter(pl.col('id').is_in(vehicle_ids))
  .vehicle.with_reported_gears()
  .select('id', 'trany', 'reported_gears')
)

In [ ]:
# noqa: E402
from vehicle_tools import (
    reported_gears, reported_gears_scalar, reported_gears_regex,
)

expected_gears = (vehicles
  .select(reported_gears=gears_expr('trany'))
)
for expression in [
    pl.col('trany').map_elements(parse_gears, return_dtype=pl.UInt32),
    pl.col('trany').map_elements(reported_gears_scalar,
    return_dtype=pl.UInt32),
    reported_gears('trany'),
    reported_gears_regex('trany'),
]:
    assert_frame_equal(
        vehicles
          .select(reported_gears=expression), expected_gears,
    )

In [ ]:
%%timeit
(vehicles
  .select(pl.col('trany').map_elements(parse_gears,
      return_dtype=pl.UInt32))
)

In [ ]:
%%timeit
(vehicles
  .select(gears_expr('trany'))
)

In [ ]:
%%timeit
(vehicles
  .select(pl.col('trany').map_elements(reported_gears_scalar,
      return_dtype=pl.UInt32))
)

In [ ]:
%%timeit
(vehicles
  .select(reported_gears('trany'))
)

In [ ]:
%%timeit
(vehicles
  .select(reported_gears_regex('trany'))
)

In [ ]:
EARTH_RADIUS_KM = 6371.0088
RELATIVE_TOLERANCE = 1e-12
ONE_MICROMETER_KM = 1 / 1_000_000_000


def haversine_python(lat1, lon1, lat2, lon2):
    coordinates = [lat1, lon1, lat2, lon2]
    if any(value is None
    or not math.isfinite(value) for value in coordinates):
        return None
    if not (-90 <= lat1 <= 90 and -90 <= lat2 <= 90):
        return None
    if not (-180 <= lon1 <= 180 and -180 <= lon2 <= 180):
        return None
    phi1, lambda1, phi2, lambda2 = map(math.radians, coordinates)
    a = (math.sin((phi2 - phi1) / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin((lambda2
         - lambda1) / 2) ** 2)
    return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(max(0.0, min(1.0,
    a))))


pairs = pl.DataFrame({
    'leg': ['same point', 'quarter equator', 'date line', 'missing',
    'bad latitude'],
    'lat1': [0.0, 0.0, 0.0, None, 91.0],
    'lon1': [0.0, 0.0, 179.0, 0.0, 0.0],
    'lat2': [0.0, 0.0, 0.0, 0.0, 0.0],
    'lon2': [0.0, 90.0, -179.0, 90.0, 0.0],
})
coordinate_columns = ['lat1', 'lon1', 'lat2', 'lon2']
python_distance = pl.struct(coordinate_columns).map_elements(
    lambda row: haversine_python(
        *(row[name] for name in coordinate_columns)),
    return_dtype=pl.Float64,
)
print(pairs
  .select('leg', distance_km=python_distance)
)

In [ ]:
def haversine_expr(lat1, lon1, lat2, lon2):
    coordinates = [pl.col(name).cast(pl.Float64) for name in [lat1, lon1,
    lat2, lon2]]
    a_lat, a_lon, b_lat, b_lon = coordinates
    valid = (
        pl.all_horizontal([column.is_finite() for column in coordinates])
        & a_lat.is_between(-90, 90) & b_lat.is_between(-90, 90)
        & a_lon.is_between(-180, 180) & b_lon.is_between(-180, 180)
    )
    phi1, lambda1, phi2, lambda2 = [column * (math.pi / 180) 
                                    for column in coordinates]
    a = (((phi2 - phi1) / 2).sin().pow(2)
         + phi1.cos() * phi2.cos() * ((lambda2 - lambda1) / 2).sin()
         .pow(2))
    return (pl.when(valid)
        .then(2 * EARTH_RADIUS_KM * a.clip(0, 1).sqrt().arcsin())
        .otherwise(None))


assert_frame_equal(
    pairs
      .select(distance_km=python_distance), 
    pairs
      .select(distance_km=haversine_expr(*coordinate_columns)),
    rel_tol=RELATIVE_TOLERANCE, abs_tol=ONE_MICROMETER_KM,
)
print(pairs
  .select('leg', distance_km=haversine_expr(*coordinate_columns))
)

In [ ]:
def haversine_numpy(columns):
    coordinates = [column.cast(pl.Float64).to_numpy() 
                   for column in columns]
    lat1, lon1, lat2, lon2 = coordinates
    valid = np.logical_and.reduce([np.isfinite(column) 
                                   for column in coordinates])
    valid &= (np.abs(lat1) <= 90) & (np.abs(lat2) <= 90)
    valid &= (np.abs(lon1) <= 180) & (np.abs(lon2) <= 180)
    safe = [np.where(valid, column, 0.0) for column in coordinates]
    phi1, lambda1, phi2, lambda2 = [np.radians(column) 
                                    for column in safe]
    a = (np.sin((phi2 - phi1) / 2) ** 2
         + np.cos(phi1) * np.cos(phi2) * np.sin((lambda2
         - lambda1) / 2) ** 2)
    distance = 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return pl.Series(np.where(valid, distance, np.nan), nan_to_null=True)


assert_frame_equal(
    pairs
      .select(distance_km=haversine_expr(*coordinate_columns)), 
    pairs
      .select(distance_km=pl.map_batches(
          coordinate_columns, haversine_numpy,
          return_dtype=pl.Float64, is_elementwise=True)),
    rel_tol=RELATIVE_TOLERANCE, abs_tol=ONE_MICROMETER_KM,
)
print(pairs
  .select('leg', distance_km=pl.map_batches(
      coordinate_columns, haversine_numpy,
      return_dtype=pl.Float64, is_elementwise=True))
)

In [ ]:
from vehicle_tools import haversine_rust_simd

simd_distance = haversine_rust_simd(*coordinate_columns)
assert_frame_equal(
    pairs.select(distance_km=haversine_expr(*coordinate_columns)),
    pairs.select(distance_km=simd_distance),
    rel_tol=RELATIVE_TOLERANCE, abs_tol=ONE_MICROMETER_KM,
)
print(pairs
 .select('leg', distance_km=simd_distance)
)

In [ ]:
chunked_pairs = pl.concat([pairs, pairs], rechunk=False).slice(1, 8)
rechunked_pairs = chunked_pairs.rechunk()
assert_frame_equal(chunked_pairs, rechunked_pairs)
print(pl.DataFrame({
    'layout': ['chunked', 'rechunked'],
    **{name: [chunked_pairs[name].n_chunks(),
    rechunked_pairs[name].n_chunks()]
       for name in coordinate_columns},
}))

In [ ]:
expected_distance = (chunked_pairs
  .select(distance_km=python_distance)
)
for expression in [haversine_expr(*coordinate_columns),
    pl.map_batches(coordinate_columns, haversine_numpy,
        return_dtype=pl.Float64, is_elementwise=True),
    rust_scalar_distance, rust_plugin_distance, simd_distance]:
    for engine in ['in-memory', 'streaming']:
        assert_frame_equal(
            chunked_pairs
              .lazy()
              .select(distance_km=expression)
              .collect(engine=engine),
            expected_distance, rel_tol=RELATIVE_TOLERANCE,
            abs_tol=ONE_MICROMETER_KM,
        )
print(expected_distance)

In [ ]:
distance_benchmark = pl.concat([pairs] * 2_000, rechunk=True)

In [ ]:
%%timeit
(distance_benchmark
  .select(distance_km=pl.struct(coordinate_columns).map_elements(
      lambda row: haversine_python(
          *(row[name] for name in coordinate_columns)),
      return_dtype=pl.Float64))
)

In [ ]:
%%timeit
(distance_benchmark
  .select(distance_km=pl.struct(coordinate_columns).map_elements(
      lambda row: haversine_rust_scalar(
          *(row[name] for name in coordinate_columns)),
      return_dtype=pl.Float64))
)

In [ ]:
%%timeit
(distance_benchmark
  .select(distance_km=haversine_rust_plugin(*coordinate_columns))
)

In [ ]:
%%timeit
(distance_benchmark
  .select(distance_km=haversine_rust_simd(*coordinate_columns))
)

In [ ]:
%%timeit
(distance_benchmark
  .select(distance_km=haversine_expr(*coordinate_columns))
)

In [ ]:
%%timeit
(distance_benchmark
  .select(distance_km=pl.map_batches(
      coordinate_columns, haversine_numpy,
      return_dtype=pl.Float64, is_elementwise=True))
)